In [ ]:
%pip install -q "datasets>=3.0,<5" "bert-score>=0.3.13,<0.4" tqdm ijson

In [ ]:
import os, json, math, random, hashlib, zipfile, gc, warnings, sys, shutil
from pathlib import Path
from collections import Counter, defaultdict
from getpass import getpass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm
from datasets import load_dataset

RUN_SMOKE_TEST = True
RUN_TRAINING = True
RUN_LOCAL_GENERATION_EVAL = True
RUN_OFFICIAL_GENERATION = True

LOCAL_EVAL_EXAMPLES = 48
CHECKPOINT_BERTSCORE_EXAMPLES = 16
DECODING_TUNING_EXAMPLES = 32
MAX_TEST_EXAMPLES = None
ALLOW_CONTEXT_TRUNCATION = False
CONTEXT_TARGET_PERCENTILE = 100.0
CONTEXT_SAFETY_MARGIN_TOKENS = 64

DATASET_ID = "Exploration-Lab/CS-781-Capstone"
DATASET_CONFIG = "phase_1"
SEED = 42

MAX_TRAIN_DOCS = 200_000
MAX_CORPUS_CHARS = 200_000_000
MAX_CHARS_PER_DOC = 50_000
HF_FS_BLOCK_SIZE = 1 * 1024 * 1024
SOURCE_FILES = {
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}

MAX_CACHED_WORD_ENCODINGS = 100_000
TARGET_VOCAB_SIZE = 32_768
TRAIN_TOKEN_BUDGET = 20_000_000
VALID_TOKEN_BUDGET = 300_000
REQUIRED_TRAIN_COVERAGE_TOKENS = 3_500_000
VALID_FRACTION = 0.02

FIXED_CONTEXT_LENGTH = 4_096
BLOCK_SIZE = FIXED_CONTEXT_LENGTH
ENCODER_MAX_LENGTH = FIXED_CONTEXT_LENGTH
DECODER_MAX_LENGTH = FIXED_CONTEXT_LENGTH
TRAIN_SOURCE_TOKENS = 1_536
TRAIN_TARGET_TOKENS = 512
TRAIN_SOURCE_WORDS = 512
TRAIN_TARGET_WORDS = 128
MIN_SOURCE_WORDS = 128
PAIR_STRIDE_WORDS = 256

TRAIN_PAIR_TOKEN_BUDGET = 36_000_000
VALID_PAIR_TOKEN_BUDGET = 500_000
GENERATION_TOKEN_BUDGETS = (2_048, 3_200)
MAX_GENERATION_TOKENS = max(GENERATION_TOKEN_BUDGETS)
MAX_BPE_TOKENS_PER_WORD = 24
OUTPUT_WORDS = 128
TEMPERATURE = 0.5
TOP_K = 50
TOP_P = 0.90
USE_KV_CACHE = True
DECODING_TEMPERATURES = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
RUN_DECODING_SWEEP = True

EPOCHS = 2
MICRO_BATCH_SIZE = 1
GRAD_ACCUM = 16
EVAL_EVERY_STEPS = 500
MAX_EVAL_BATCHES = 256
WARMUP_FRACTION = 0.05
LEARNING_RATE = 2.5e-4
WARM_START_LEARNING_RATE = 1e-4
CONTINUE_FROM_EXISTING_CHECKPOINT = True
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
WORKERS = 0
USE_GRAD_CHECKPOINTING = False

USE_SWIGLU = True
USE_COPY_HEAD = True
ARCHITECTURE_NAME = "encoder_decoder_transformer_rope_gqa_rmsnorm_swiglu_copyhead_t4_reused_bpe32k_ctx4k"
MODEL_D = 768
MODEL_HEADS = 12
MODEL_KV_HEADS = 4
ENCODER_LAYERS = 7
DECODER_LAYERS = 7
MODEL_FF = 2752
MODEL_DROPOUT = 0.0
RMS_NORM_EPS = 1e-6
ROPE_BASE = 10_000.0

CACHE_DIR = Path("phase1_encoder_decoder_t4_reused_bpe32k_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path("phase1_encoder_decoder_gqa_rmsnorm_t4_reused_bpe32k_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_PATH = CACHE_DIR / f"legal_custom_bytebpe_{TARGET_VOCAB_SIZE}.json"
LATEST_CHECKPOINT_PATH = OUTPUT_DIR / "latest_t4_reused_bpe32k_checkpoint.pt"
BEST_CHECKPOINT_PATH = OUTPUT_DIR / "best_t4_reused_bpe32k_checkpoint.pt"
PACKAGED_TOKENIZER_PATH = OUTPUT_DIR / f"legal_custom_bytebpe_{TARGET_VOCAB_SIZE}.json"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
elif RUN_TRAINING:
    raise RuntimeError("Enable the Kaggle GPU accelerator before running this notebook.")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass("Hugging Face read token: ")
if not HF_TOKEN:
    raise RuntimeError("A Hugging Face token is required to access the gated Phase-1 dataset.")
def stable_json_hash(obj):
    payload = json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, allow_nan=False)

def make_grad_scaler(enabled):
    if not enabled:
        return None
    try:
        return torch.amp.GradScaler("cuda", enabled=True)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=True)

print("Requested unique train-token budget:", f"{TRAIN_TOKEN_BUDGET:,}")
print("Minimum training-pair corpus coverage:", f"{REQUIRED_TRAIN_COVERAGE_TOKENS:,}")
print("Tokenizer:", TARGET_VOCAB_SIZE, "custom ByteBPE entries; loading an existing JSON, no fitting.")
print("Runtime note: these are bounded T4-oriented settings; actual runtime depends on Kaggle GPU speed and data access.")
print("Raw validation-token sample budget:", f"{VALID_TOKEN_BUDGET:,}")
print("Sequence-pair token budgets:", f"{TRAIN_PAIR_TOKEN_BUDGET:,}", f"/ {VALID_PAIR_TOKEN_BUDGET:,}")
print("Training schedule:", EPOCHS, "epoch(s), gradient accumulation", GRAD_ACCUM)

In [ ]:
from huggingface_hub import HfFileSystem
import ijson

hffs = HfFileSystem(token=HF_TOKEN, block_size=HF_FS_BLOCK_SIZE)

def iter_source_records(source_path):
    remote_path = f"datasets/{DATASET_ID}/{source_path}"
    with hffs.open(remote_path, "rb") as fh:

        yield from ijson.items(fh, "item")

def normalize_train_text(record):
    if not isinstance(record, dict) or "text" not in record:
        raise RuntimeError(
            "Unexpected Phase-1 record: every training record must contain a 'text' field."
        )
    text = record["text"]
    if not isinstance(text, str):
        return ""
    return text.strip()

def is_validation_document(source_name, source_row_index):
    key = f"{SEED}|{source_name}|{source_row_index}".encode("utf-8")
    digest = hashlib.blake2b(key, digest_size=8).digest()
    bucket = int.from_bytes(digest, byteorder="little", signed=False) % 10_000
    threshold = int(round(VALID_FRACTION * 10_000))
    return bucket < threshold

def iter_bounded_source_documents(source_name):
    source_path = SOURCE_FILES[source_name]
    source_char_budget = (
        None if MAX_CORPUS_CHARS is None
        else max(1, MAX_CORPUS_CHARS // len(SOURCE_FILES))
    )
    source_doc_budget = (
        None if MAX_TRAIN_DOCS is None
        else max(1, MAX_TRAIN_DOCS // len(SOURCE_FILES))
    )

    char_count = 0
    doc_count = 0

    for row_index, record in enumerate(iter_source_records(source_path)):
        if source_doc_budget is not None and doc_count >= source_doc_budget:
            break
        if source_char_budget is not None and char_count >= source_char_budget:
            break

        text = normalize_train_text(record)
        if not text:
            continue

        if MAX_CHARS_PER_DOC is not None and len(text) > MAX_CHARS_PER_DOC:
            text = text[:MAX_CHARS_PER_DOC]

        if source_char_budget is not None:
            remaining_chars = source_char_budget - char_count
            if len(text) > remaining_chars:
                text = text[:remaining_chars]
        else:
            remaining_chars = None

        if not text:
            break

        is_valid = is_validation_document(source_name, row_index)
        doc_count += 1
        char_count += len(text)
        yield source_name, row_index, text, is_valid

        if remaining_chars is not None and len(text) >= remaining_chars:
            break

def allocate_source_token_budgets(total_budget):
    sizes = {}
    for source_name, source_path in SOURCE_FILES.items():
        info = hffs.info(f"datasets/{DATASET_ID}/{source_path}")
        sizes[source_name] = max(1, int(info.get("size", 1)))
    size_total = sum(sizes.values())
    exact = {
        name: total_budget * size / size_total
        for name, size in sizes.items()
    }
    budgets = {name: int(value) for name, value in exact.items()}
    remainder = int(total_budget - sum(budgets.values()))
    order = sorted(
        SOURCE_FILES,
        key=lambda name: exact[name] - budgets[name],
        reverse=True,
    )
    for name in order[:remainder]:
        budgets[name] += 1
    return budgets, sizes

load_audit = {
    "dataset_id": DATASET_ID,
    "dataset_config": DATASET_CONFIG,
    "split": "train",
    "seed": SEED,
    "loader": "HfFileSystem + ijson incremental JSON parsing",
    "source_files": SOURCE_FILES,
    "source_char_budget": MAX_CORPUS_CHARS,
    "source_doc_budget": MAX_TRAIN_DOCS,
    "max_tokens_per_document": MAX_CHARS_PER_DOC,
    "max_chars_per_doc": MAX_CHARS_PER_DOC,
    "validation_fraction": VALID_FRACTION,
}

print("Phase-1 training loader: each official JSON source is parsed incrementally.")
print("Global document/character caps:", MAX_TRAIN_DOCS, "/", MAX_CORPUS_CHARS)
print("Maximum characters retained per document:", MAX_CHARS_PER_DOC)

for source_name in SOURCE_FILES:
    count = 0
    chars = 0
    valid = 0
    truncated = 0
    for _, row_index, text, is_valid in iter_bounded_source_documents(source_name):
        count += 1
        chars += len(text)
        valid += int(is_valid)
        if len(text) == MAX_CHARS_PER_DOC:
            truncated += 1
        if count >= 3:

            break
    print(f"{source_name}: first 3 parsed records OK | sample_chars={chars:,} | sample_valid={valid}")

save_json(OUTPUT_DIR / "training_document_load_metadata.json", load_audit)

In [ ]:
split_metadata = {
    "seed": SEED,
    "validation_fraction": VALID_FRACTION,
    "split_unit": "document",
    "storage": "streamed; not materialized in RAM",
    "source_files": SOURCE_FILES,
    "source_doc_budget_each": MAX_TRAIN_DOCS,
    "source_char_budget_each": MAX_CORPUS_CHARS,
}
save_json(OUTPUT_DIR / "data_split_metadata.json", split_metadata)
print(split_metadata)

In [ ]:
class ByteBPE:
    def __init__(self, vocab_size=5000):
        self.target_vocab_size = int(vocab_size)
        self.specials = ["<pad>", "<unk>", "<s>", "</s>"]
        self.eow = "</w>"
        self.merges = []
        self.itos = []
        self.stoi = {}
        self.payload = {}
        self._encode_cache = {}
        self.merge_ranks = {}
        self.vocab_size = 0

    def _init_symbols(self):
        self.itos = self.specials + [f"<0x{i:02X}>" for i in range(256)] + [self.eow]
        self.stoi = {symbol: i for i, symbol in enumerate(self.itos)}
        self.payload = {i: bytes([i - 4]) for i in range(4, 260)}
        self.payload[260] = b""
        self.vocab_size = len(self.itos)
        self.merge_ranks = {}

    @staticmethod
    def _merge_sequence(sequence, pair, merged_id):
        out = []
        i = 0
        while i < len(sequence):
            if (
                i + 1 < len(sequence)
                and sequence[i] == pair[0]
                and sequence[i + 1] == pair[1]
            ):
                out.append(merged_id)
                i += 2
            else:
                out.append(sequence[i])
                i += 1
        return out

    def encode_word(self, word):
        word = str(word)
        cached = self._encode_cache.get(word)
        if cached is not None:
            return list(cached)

        sequence = [byte + 4 for byte in word.encode("utf-8")] + [260]

        while len(sequence) > 1:
            best_rank = None
            best_pair = None
            for i in range(len(sequence) - 1):
                candidate = (sequence[i], sequence[i + 1])
                rank = self.merge_ranks.get(candidate)
                if rank is not None and (best_rank is None or rank < best_rank):
                    best_rank = rank
                    best_pair = candidate
            if best_pair is None:
                break
            sequence = self._merge_sequence(
                sequence, best_pair, 261 + best_rank
            )

        if len(self._encode_cache) < MAX_CACHED_WORD_ENCODINGS:
            self._encode_cache[word] = tuple(sequence)
        return sequence

    def encode(self, text):
        return [token_id for word in str(text).split() for token_id in self.encode_word(word)]

    def to_json(self):
        return {
            "merges": [list(pair) for pair in self.merges],
            "itos": self.itos,
            "vocab_size": self.vocab_size,
        }

    @classmethod
    def from_json(cls, payload):
        tokenizer = cls(vocab_size=int(payload["vocab_size"]))
        tokenizer._init_symbols()
        tokenizer.merges = [tuple(map(int, pair)) for pair in payload["merges"]]
        for pair in tokenizer.merges:
            new_id = len(tokenizer.itos)
            symbol = f"<{pair[0]}+{pair[1]}>"
            tokenizer.itos.append(symbol)
            tokenizer.stoi[symbol] = new_id
            tokenizer.payload[new_id] = (
                tokenizer.payload.get(pair[0], b"")
                + tokenizer.payload.get(pair[1], b"")
            )
        tokenizer.vocab_size = len(tokenizer.itos)
        tokenizer.merge_ranks = {
            pair: rank for rank, pair in enumerate(tokenizer.merges)
        }
        tokenizer._encode_cache = {}
        if tokenizer.itos != payload.get("itos"):
            raise ValueError("Saved ByteBPE vocabulary and merge list are inconsistent.")
        return tokenizer

def build_word_end_flags(bpe):
    ends_word = [False] * len(bpe.itos)
    ends_word[260] = True
    for rank, (left, right) in enumerate(bpe.merges):
        ends_word[261 + rank] = ends_word[left] or ends_word[right]
    return ends_word

def decode_bpe_ids(bpe, token_ids, skip_special_tokens=True):
    ends_word = build_word_end_flags(bpe)
    pieces = []
    for token_id in token_ids:
        token_id = int(token_id)
        if token_id < 0 or token_id >= len(bpe.itos):
            continue
        if token_id in (0, 1, 2, 3):
            if skip_special_tokens:
                continue
            pieces.append(bpe.itos[token_id])
            continue
        payload = bpe.payload.get(token_id, b"")
        pieces.append(payload.decode("utf-8", errors="replace"))
        if ends_word[token_id]:
            pieces.append(" ")
    return "".join(pieces).strip()

def find_existing_tokenizer_file():
    candidates = []
    explicit_path = os.environ.get("LEGAL_BPE_TOKENIZER_PATH")
    if explicit_path:
        candidates.append(Path(explicit_path).expanduser())

    candidates.extend([

        Path("legal_custom_bytebpe_32768.json"),
        Path("legal_bpe_tokenizer_32768.json"),
        Path("legal_bpe_tokenizer.json"),
        Path("legal_bpe_tokenizer (2).json"),
        Path("legal_bpe_tokenizer (3).json"),

        Path("phase1_encoder_decoder_1b_custom_bpe_cache/legal_custom_bytebpe_32768.json"),
        Path("phase1_encoder_decoder_gqa_rmsnorm_1b_outputs/legal_custom_bytebpe_32768.json"),
        Path("/kaggle/working/legal_custom_bytebpe_32768.json"),
        Path("/kaggle/working/legal_bpe_tokenizer.json"),
        Path("/kaggle/input/legal_custom_bytebpe_32768.json"),

        TOKENIZER_PATH,
    ])

    downloads = Path.home() / "Downloads"
    if downloads.exists():
        candidates.extend(sorted(downloads.glob("*tokenizer*.json")))
        candidates.extend(sorted(downloads.glob("*bytebpe*.json")))

    cwd = Path.cwd()
    if cwd.exists():
        candidates.extend(sorted(cwd.glob("*tokenizer*.json")))
        candidates.extend(sorted(cwd.glob("*bytebpe*.json")))

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend(sorted(kaggle_input.rglob("*tokenizer*.json")))
        candidates.extend(sorted(kaggle_input.rglob("*bytebpe*.json")))

    seen = set()
    failures = []
    for candidate in candidates:
        try:
            marker = str(candidate.resolve())
        except Exception:
            marker = str(candidate)
        if marker in seen:
            continue
        seen.add(marker)
        if not candidate.is_file():
            continue
        try:
            payload = json.loads(candidate.read_text(encoding="utf-8"))
            if int(payload.get("vocab_size", -1)) != TARGET_VOCAB_SIZE:
                failures.append(
                    f"{candidate}: vocab_size={payload.get('vocab_size')}"
                )
                continue
            loaded = ByteBPE.from_json(payload)
            if loaded.vocab_size != TARGET_VOCAB_SIZE:
                failures.append(
                    f"{candidate}: reconstructed vocab_size={loaded.vocab_size}"
                )
                continue
            return candidate, loaded
        except Exception as exc:
            failures.append(f"{candidate}: {exc}")

    details = "\n".join(failures[:8])
    raise FileNotFoundError(
        "Could not find a valid 32,768-entry custom ByteBPE JSON. "
        "In Kaggle, attach the downloaded tokenizer with Add Input, or set "
        "LEGAL_BPE_TOKENIZER_PATH to its exact path. This notebook will not "
        "train a replacement tokenizer."
        + (f"\nCandidate files checked:\n{details}" if details else "")
    )

TOKENIZER_PATH, tokenizer = find_existing_tokenizer_file()
PACKAGED_TOKENIZER_PATH.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(TOKENIZER_PATH, PACKAGED_TOKENIZER_PATH)
TOKENIZER_SHA256 = hashlib.sha256(TOKENIZER_PATH.read_bytes()).hexdigest()
print("Loaded downloaded custom ByteBPE; tokenizer training is disabled.")
print("Tokenizer file:", TOKENIZER_PATH)
print("Tokenizer SHA256:", TOKENIZER_SHA256)

PAD_ID = tokenizer.stoi["<pad>"]
UNK_ID = tokenizer.stoi["<unk>"]
BOS_ID = tokenizer.stoi["<s>"]
EOS_ID = tokenizer.stoi["</s>"]
EOW_ID = 260
SPECIAL_IDS = {PAD_ID, UNK_ID, BOS_ID, EOS_ID}
VOCAB_SIZE = tokenizer.vocab_size

if [tokenizer.itos[i] for i in range(4)] != ["<pad>", "<unk>", "<s>", "</s>"]:
    raise RuntimeError("Custom ByteBPE reserved-token ordering changed unexpectedly.")
if EOW_ID >= VOCAB_SIZE:
    raise RuntimeError("Custom ByteBPE end-of-word symbol is missing.")
if VOCAB_SIZE < 32_000:
    raise RuntimeError(f"Tokenizer vocabulary is unexpectedly small: {VOCAB_SIZE:,}")

print("Custom ByteBPE vocab size:", VOCAB_SIZE)
print("Reserved IDs:", {"pad": PAD_ID, "unk": UNK_ID, "bos": BOS_ID, "eos": EOS_ID})

In [ ]:
TEXT_KEYS = (
    "text", "content", "document", "judgment", "judgement", "body",
    "full_text", "prompt", "prompt_text", "input", "case_text",
)

def extract_text(record):
    if isinstance(record, str):
        return record
    if isinstance(record, dict):
        for key in TEXT_KEYS:
            value = record.get(key)
            if isinstance(value, str) and value.strip():
                return value
        for key, value in record.items():
            if str(key).lower() in {"id", "document_id", "doc_id", "example_id"}:
                continue
            if isinstance(value, str) and len(value.strip()) > 20:
                return value
            if isinstance(value, (dict, list, tuple)):
                found = extract_text(value)
                if found:
                    return found
    if isinstance(record, (list, tuple)):
        for value in record:
            found = extract_text(value)
            if found:
                return found
    return ""

def extract_id(record, fallback):
    if isinstance(record, dict):
        for key in ("id", "ID", "example_id", "document_id", "doc_id"):
            value = record.get(key)
            if value is not None and not isinstance(value, (dict, list)):
                return str(value)

        for key, value in record.items():
            if str(key).strip().isdigit() and isinstance(value, dict) and extract_text(value):
                return str(key)
    return str(fallback)

test_stream = load_dataset(
    DATASET_ID,
    DATASET_CONFIG,
    split="test",
    streaming=True,
    token=HF_TOKEN,
)
test_examples = []
stop_loading = False

for row_index, row in enumerate(
    tqdm(test_stream, desc="Reading official Phase-1 test prompts")
):

    if (
        isinstance(row, dict)
        and row
        and all(str(key).isdigit() for key in row)
    ):
        records = ((str(key), value) for key, value in row.items())
        if row_index == 0:
            print(
                f"Found ID-keyed test object with {len(row)} entries; "
                f"sample IDs: {list(row)[:5]}"
            )
    else:

        records = ((extract_id(row, row_index + 1), row),)
        if row_index == 0:
            fields = list(row.keys())[:10] if isinstance(row, dict) else type(row).__name__
            print("First official test-row fields:", fields)

    for official_id, record in records:
        prompt = extract_text(record).strip()
        if not prompt:
            fields = list(record.keys()) if isinstance(record, dict) else type(record).__name__
            raise ValueError(
                f"Could not extract prompt for official test ID {official_id}; "
                f"available fields={fields}"
            )

        test_examples.append({"id": str(official_id), "prompt": prompt})

        if (
            MAX_TEST_EXAMPLES is not None
            and len(test_examples) >= MAX_TEST_EXAMPLES
        ):
            stop_loading = True
            break

    if stop_loading:
        break

print(f"Loaded official test prompts: {len(test_examples)}")
print("First IDs:", [item["id"] for item in test_examples[:5]])
print("Last IDs:", [item["id"] for item in test_examples[-5:]])

if not test_examples:
    raise RuntimeError("No official test prompts loaded.")

official_ids = [item["id"] for item in test_examples]
if len(set(official_ids)) != len(official_ids):
    raise RuntimeError("Duplicate official test IDs detected.")

prompt_audit = []
for item in test_examples:
    tok_len = len(tokenizer.encode(item["prompt"]))
    prompt_audit.append({
        "id": item["id"],
        "words": len(item["prompt"].split()),
        "tokens": tok_len,
    })

save_json(OUTPUT_DIR / "official_test_prompt_length_audit.json", prompt_audit)

prompt_lengths = [row["tokens"] for row in prompt_audit]
print(f"Official test examples: {len(test_examples)}")
print(f"Prompt words: min={min(row['words'] for row in prompt_audit)}, max={max(row['words'] for row in prompt_audit)}")
print(f"Prompt tokens: min={min(prompt_lengths)}, mean={np.mean(prompt_lengths):.1f}, max={max(prompt_lengths)}")

if max(prompt_lengths) + 2 > ENCODER_MAX_LENGTH:
    offending = max(prompt_lengths) + 2
    raise RuntimeError(
        f"An official prompt plus BOS/EOS requires {offending} encoder tokens; "
        f"ENCODER_MAX_LENGTH={ENCODER_MAX_LENGTH}. No silent prompt truncation is used."
    )

validation_context_samples = []
MAX_CONTEXT_SAMPLE_DOCS_PER_SOURCE = 64
MAX_CONTEXT_SAMPLE_CHARS = 12_000
for source_name in SOURCE_FILES:
    collected = 0
    for _, _, text, is_valid in iter_bounded_source_documents(source_name):
        if not is_valid:
            continue
        validation_context_samples.append(text[:MAX_CONTEXT_SAMPLE_CHARS])
        collected += 1
        if collected >= MAX_CONTEXT_SAMPLE_DOCS_PER_SOURCE:
            break

heldout_validation_pairs = []
completion_lengths = []
for text in validation_context_samples:
    words = text.split()
    if len(words) >= 512 + OUTPUT_WORDS:
        prompt = " ".join(words[:512])
        reference = " ".join(words[512:512 + OUTPUT_WORDS])
        heldout_validation_pairs.append((prompt, reference))
        completion_lengths.append(len(tokenizer.encode(reference)))
    if len(completion_lengths) >= max(256, LOCAL_EVAL_EXAMPLES):
        break

random.Random(SEED).shuffle(heldout_validation_pairs)
required_heldout = (
    CHECKPOINT_BERTSCORE_EXAMPLES
    + DECODING_TUNING_EXAMPLES
    + LOCAL_EVAL_EXAMPLES
)
if len(heldout_validation_pairs) < required_heldout:
    raise RuntimeError(
        f"Need {required_heldout} held-out continuations for the three disjoint sets; "
        f"collected {len(heldout_validation_pairs)}."
    )
CHECKPOINT_SELECTION_PAIRS = heldout_validation_pairs[:CHECKPOINT_BERTSCORE_EXAMPLES]
start = CHECKPOINT_BERTSCORE_EXAMPLES
stop = start + DECODING_TUNING_EXAMPLES
DECODING_TUNING_PAIRS = heldout_validation_pairs[start:stop]
HELDOUT_VALIDATION_PAIRS = heldout_validation_pairs[stop:stop + LOCAL_EVAL_EXAMPLES]
print("Disjoint held-out sets:", {
    "checkpoint_selection": len(CHECKPOINT_SELECTION_PAIRS),
    "decoding_tuning": len(DECODING_TUNING_PAIRS),
    "final_local_report": len(HELDOUT_VALIDATION_PAIRS),
})
del heldout_validation_pairs
del validation_context_samples
gc.collect()

if completion_lengths:
    estimated_completion = int(np.percentile(completion_lengths, CONTEXT_TARGET_PERCENTILE))
    estimated_completion = max(estimated_completion, 256)
    print(
        f"Validation 128-word target token estimate: "
        f"median={int(np.median(completion_lengths))}, "
        f"p{CONTEXT_TARGET_PERCENTILE:g}={estimated_completion}, "
        f"max={max(completion_lengths)}"
    )
else:
    estimated_completion = 256
    print("Could not estimate target token length; using conservative 256-token estimate.")

CONTEXT_ESTIMATED_COMPLETION_TOKENS = estimated_completion
if CONTEXT_ESTIMATED_COMPLETION_TOKENS > DECODER_MAX_LENGTH:
    raise RuntimeError(
        f"Estimated completion length {CONTEXT_ESTIMATED_COMPLETION_TOKENS} exceeds "
        f"DECODER_MAX_LENGTH={DECODER_MAX_LENGTH}."
    )
if MAX_GENERATION_TOKENS > DECODER_MAX_LENGTH:
    raise RuntimeError(
        f"Generation budget {MAX_GENERATION_TOKENS} exceeds "
        f"DECODER_MAX_LENGTH={DECODER_MAX_LENGTH}."
    )

save_json(OUTPUT_DIR / "context_safety_audit.json", {
    "encoder_max_length": int(ENCODER_MAX_LENGTH),
    "decoder_max_length": int(DECODER_MAX_LENGTH),
    "max_official_prompt_tokens": int(max(prompt_lengths)),
    "prompt_special_tokens": 2,
    "estimated_completion_tokens": int(CONTEXT_ESTIMATED_COMPLETION_TOKENS),
    "generation_token_budget": int(MAX_GENERATION_TOKENS),
    "source_and_target_contexts_are_independent": True,
    "passes": bool(
        max(prompt_lengths) + 2 <= ENCODER_MAX_LENGTH
        and MAX_GENERATION_TOKENS <= DECODER_MAX_LENGTH
    ),
})

In [ ]:
def encode_streamed_split(is_validation, token_budget, label):
    source_budgets, source_sizes = allocate_source_token_budgets(token_budget)
    source_arrays = []
    split_name = "validation" if is_validation else "train"
    if "stream_split_document_counts" not in globals():
        globals()["stream_split_document_counts"] = {"train": {}, "validation": {}}
    if "stream_split_source_ranges" not in globals():
        globals()["stream_split_source_ranges"] = {"train": {}, "validation": {}}

    offset = 0
    for source_name in SOURCE_FILES:
        per_source_budget = source_budgets[source_name]
        ids = np.empty(per_source_budget, dtype=np.int32)
        used = 0
        doc_count = 0
        for _, _, text, row_is_validation in tqdm(
            iter_bounded_source_documents(source_name),
            desc=f"{label}: {source_name}",
        ):
            if row_is_validation != is_validation:
                continue
            remaining = per_source_budget - used
            if remaining <= 1:
                break
            encoded = tokenizer.encode(text)
            if not encoded:
                continue
            take = min(len(encoded), remaining - 1)
            ids[used:used + take] = np.asarray(encoded[:take], dtype=np.int32)
            used += take
            ids[used] = EOS_ID
            used += 1
            doc_count += 1
            if used >= per_source_budget:
                break

        source_arrays.append(ids[:used].copy())
        stream_split_document_counts[split_name][source_name] = int(doc_count)
        stream_split_source_ranges[split_name][source_name] = [int(offset), int(offset + used)]
        offset += used
        print(f"{label} | {source_name}: docs={doc_count:,}, tokens={used:,}")

    return np.concatenate(source_arrays).astype(np.int32, copy=False)

ENCODED_CACHE_DIR = CACHE_DIR / "encoded_training_data"
ENCODED_CACHE_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_IDS_CACHE = ENCODED_CACHE_DIR / "train_ids.npy"
VALID_IDS_CACHE = ENCODED_CACHE_DIR / "valid_ids.npy"
ENCODED_CACHE_METADATA = ENCODED_CACHE_DIR / "metadata.json"
ENCODED_DATA_SPEC = {
    "dataset_id": DATASET_ID,
    "dataset_config": DATASET_CONFIG,
    "source_files": SOURCE_FILES,
    "seed": SEED,
    "validation_fraction": VALID_FRACTION,
    "max_train_docs": MAX_TRAIN_DOCS,
    "max_corpus_chars": MAX_CORPUS_CHARS,
    "source_token_budgets": allocate_source_token_budgets(TRAIN_TOKEN_BUDGET)[0],
    "source_file_sizes": allocate_source_token_budgets(TRAIN_TOKEN_BUDGET)[1],
    "max_chars_per_doc": MAX_CHARS_PER_DOC,
    "train_token_budget": TRAIN_TOKEN_BUDGET,
    "valid_token_budget": VALID_TOKEN_BUDGET,
    "tokenizer_sha256": TOKENIZER_SHA256,
    "pair_schema_version": "seq2seq-prefix-target-v1",
}
ENCODED_DATA_SIGNATURE = stable_json_hash(ENCODED_DATA_SPEC)

cache_loaded = False
if TRAIN_IDS_CACHE.exists() and VALID_IDS_CACHE.exists() and ENCODED_CACHE_METADATA.exists():
    try:
        cache_metadata = json.loads(ENCODED_CACHE_METADATA.read_text(encoding="utf-8"))
        if (
            cache_metadata.get("signature") == ENCODED_DATA_SIGNATURE
            and "source_ranges" in cache_metadata
        ):
            train_ids = np.load(TRAIN_IDS_CACHE, mmap_mode="r")
            valid_ids = np.load(VALID_IDS_CACHE, mmap_mode="r")
            stream_split_document_counts = cache_metadata["document_counts"]
            stream_split_source_ranges = cache_metadata["source_ranges"]
            cache_loaded = True
            print("Reusing compatible memory-mapped token arrays and source ranges.")
    except Exception as exc:
        print("Encoded token cache could not be reused; rebuilding it:", exc)

if not cache_loaded:
    stream_split_document_counts = {"train": {}, "validation": {}}
    stream_split_source_ranges = {"train": {}, "validation": {}}
    train_ids = encode_streamed_split(False, TRAIN_TOKEN_BUDGET, "Encoding train")
    valid_ids = encode_streamed_split(True, VALID_TOKEN_BUDGET, "Encoding validation")

    ENCODED_CACHE_METADATA.unlink(missing_ok=True)
    train_tmp = ENCODED_CACHE_DIR / "train_ids.pending.npy"
    valid_tmp = ENCODED_CACHE_DIR / "valid_ids.pending.npy"
    np.save(train_tmp, train_ids)
    np.save(valid_tmp, valid_ids)
    train_tmp.replace(TRAIN_IDS_CACHE)
    valid_tmp.replace(VALID_IDS_CACHE)
    save_json(ENCODED_CACHE_METADATA, {
        "signature": ENCODED_DATA_SIGNATURE,
        "document_counts": stream_split_document_counts,
        "source_ranges": stream_split_source_ranges,
        "train_tokens": int(len(train_ids)),
        "valid_tokens": int(len(valid_ids)),
    })

TRAIN_DOCUMENT_COUNT = int(sum(stream_split_document_counts["train"].values()))
VALID_DOCUMENT_COUNT = int(sum(stream_split_document_counts["validation"].values()))
print("Actual train tokens:", f"{len(train_ids):,}")
print("Actual validation tokens:", f"{len(valid_ids):,}")
print("Actual train documents:", f"{TRAIN_DOCUMENT_COUNT:,}")
print("Actual validation documents:", f"{VALID_DOCUMENT_COUNT:,}")

def build_pair_specs(ids, source_ranges, total_pair_token_budget, label):
    source_token_lengths = {
        name: max(0, int(ranges[1]) - int(ranges[0]))
        for name, ranges in source_ranges.items()
    }
    source_token_total = max(1, sum(source_token_lengths.values()))
    all_specs = []
    token_count_by_source = {}
    pairs_by_source = {}
    skipped_by_source = {}
    word_ends_flags = np.asarray(build_word_end_flags(tokenizer), dtype=np.bool_)

    for source_name in SOURCE_FILES:
        source_begin, source_end = map(int, source_ranges[source_name])
        per_source_budget = max(
            1,
            int(total_pair_token_budget * source_token_lengths[source_name] / source_token_total),
        )
        source_budget = 0
        source_specs = []
        skipped_token_cap = 0
        document_start = source_begin
        eos_positions = np.flatnonzero(np.asarray(ids[source_begin:source_end]) == EOS_ID)
        eos_positions = eos_positions + source_begin

        for doc_end in eos_positions:
            doc_end = int(doc_end)
            doc_ids = np.asarray(ids[document_start:doc_end], dtype=np.int64)

            word_ends = np.flatnonzero(word_ends_flags[doc_ids]) + document_start + 1
            word_count = len(word_ends)
            if word_count >= MIN_SOURCE_WORDS + TRAIN_TARGET_WORDS:
                word_starts = np.concatenate((
                    np.asarray([document_start], dtype=np.int64),
                    word_ends[:-1].astype(np.int64, copy=False),
                ))
                last_target_start = word_count - TRAIN_TARGET_WORDS
                first_target_start = max(
                    MIN_SOURCE_WORDS,
                    min(TRAIN_SOURCE_WORDS, last_target_start),
                )
                target_starts = list(range(
                    first_target_start,
                    last_target_start + 1,
                    PAIR_STRIDE_WORDS,
                ))
                if last_target_start not in target_starts:
                    target_starts.append(last_target_start)

                for target_word_start in sorted(set(target_starts)):
                    source_word_start = max(0, target_word_start - TRAIN_SOURCE_WORDS)
                    source_token_start = int(word_starts[source_word_start])
                    source_token_end = int(word_ends[target_word_start - 1])
                    target_token_start = int(word_starts[target_word_start])
                    target_token_end = int(
                        word_ends[target_word_start + TRAIN_TARGET_WORDS - 1]
                    )
                    source_token_count = source_token_end - source_token_start + 2
                    target_token_count = target_token_end - target_token_start

                    if (
                        source_token_count > TRAIN_SOURCE_TOKENS
                        or target_token_count > TRAIN_TARGET_TOKENS
                    ):
                        skipped_token_cap += 1
                        continue
                    pair_cost = source_token_count + target_token_count
                    if source_budget + pair_cost > per_source_budget:
                        break
                    source_specs.append((
                        source_token_start, source_token_end,
                        target_token_start, target_token_end,
                    ))
                    source_budget += pair_cost

                if source_budget >= per_source_budget:
                    document_start = doc_end + 1
                    break
            document_start = doc_end + 1

        all_specs.extend(source_specs)
        token_count_by_source[source_name] = int(source_budget)
        pairs_by_source[source_name] = int(len(source_specs))
        skipped_by_source[source_name] = int(skipped_token_cap)
        print(
            f"{label} | {source_name}: pairs={len(source_specs):,}, "
            f"source+target tokens={source_budget:,}, over-cap skipped={skipped_token_cap:,}"
        )

    return all_specs, token_count_by_source, pairs_by_source, skipped_by_source

def count_unique_pair_coverage(pair_specs):
    intervals = sorted((int(spec[0]), int(spec[3])) for spec in pair_specs)
    total = 0
    current_start = None
    current_end = None
    for start, end in intervals:
        if end <= start:
            continue
        if current_start is None:
            current_start, current_end = start, end
        elif start <= current_end:
            current_end = max(current_end, end)
        else:
            total += current_end - current_start
            current_start, current_end = start, end
    if current_start is not None:
        total += current_end - current_start
    return int(total)

class Seq2SeqContinuationPairs(Dataset):
    def __init__(self, ids, pair_specs):
        self.ids = ids
        self.pair_specs = list(pair_specs)

    def __len__(self):
        return len(self.pair_specs)

    def __getitem__(self, index):
        source_start, source_end, target_start, target_end = self.pair_specs[index]
        source_content = np.asarray(self.ids[source_start:source_end], dtype=np.int64)
        target_content = np.asarray(self.ids[target_start:target_end], dtype=np.int64)
        source = np.concatenate((
            np.asarray([BOS_ID], dtype=np.int64),
            source_content,
            np.asarray([EOS_ID], dtype=np.int64),
        ))
        decoder_input = np.concatenate((
            np.asarray([BOS_ID], dtype=np.int64),
            target_content[:-1],
        ))
        labels = target_content.copy()
        return (
            torch.tensor(source, dtype=torch.long),
            torch.tensor(decoder_input, dtype=torch.long),
            torch.tensor(labels, dtype=torch.long),
        )

train_pair_specs, train_pair_tokens_by_source, train_pairs_by_source, train_pairs_skipped_by_source = build_pair_specs(
    train_ids,
    stream_split_source_ranges["train"],
    TRAIN_PAIR_TOKEN_BUDGET,
    "Train continuation pairs",
)
valid_pair_specs, valid_pair_tokens_by_source, valid_pairs_by_source, valid_pairs_skipped_by_source = build_pair_specs(
    valid_ids,
    stream_split_source_ranges["validation"],
    VALID_PAIR_TOKEN_BUDGET,
    "Validation continuation pairs",
)
random.Random(SEED).shuffle(valid_pair_specs)
train_pairs = Seq2SeqContinuationPairs(train_ids, train_pair_specs)
valid_pairs = Seq2SeqContinuationPairs(valid_ids, valid_pair_specs)
TRAIN_PAIR_TOKEN_COUNT = int(sum(train_pair_tokens_by_source.values()))
VALID_PAIR_TOKEN_COUNT = int(sum(valid_pair_tokens_by_source.values()))
TRAIN_COVERED_TOKEN_COUNT = count_unique_pair_coverage(train_pair_specs)
if TRAIN_COVERED_TOKEN_COUNT < REQUIRED_TRAIN_COVERAGE_TOKENS:
    raise RuntimeError(
        f"Training pairs cover only {TRAIN_COVERED_TOKEN_COUNT:,} unique corpus tokens; "
        f"at least {REQUIRED_TRAIN_COVERAGE_TOKENS:,} are required. "
        "Increase TRAIN_PAIR_TOKEN_BUDGET or adjust pair limits before training."
    )

if len(train_pairs) == 0 or len(valid_pairs) == 0:
    raise RuntimeError("Could not form encoder-decoder training/validation continuation pairs.")

print("Train continuation pairs:", f"{len(train_pairs):,}")
print("Validation continuation pairs:", f"{len(valid_pairs):,}")
print("Unique corpus tokens covered by train pairs:", f"{TRAIN_COVERED_TOKEN_COUNT:,}")
print("Train source+target token presentations per epoch:", f"{TRAIN_PAIR_TOKEN_COUNT:,}")
print("Validation source+target token presentations:", f"{VALID_PAIR_TOKEN_COUNT:,}")
print("Pair limits:", {
    "encoder_source_tokens_max": TRAIN_SOURCE_TOKENS,
    "decoder_target_tokens_max": TRAIN_TARGET_TOKENS,
    "source_words_max": TRAIN_SOURCE_WORDS,
    "target_words_exact": TRAIN_TARGET_WORDS,
    "minimum_source_words": MIN_SOURCE_WORDS,
    "stride_words": PAIR_STRIDE_WORDS,
})

TOKENS_PER_OPTIMIZER_STEP = max(1, int(
    round(TRAIN_PAIR_TOKEN_COUNT / max(1, len(train_pairs))) * MICRO_BATCH_SIZE * GRAD_ACCUM
))
STEPS_PER_EPOCH = max(1, math.ceil(len(train_pairs) / (MICRO_BATCH_SIZE * GRAD_ACCUM)))
MAX_STEPS = EPOCHS * STEPS_PER_EPOCH
WARMUP_STEPS = max(1, int(round(MAX_STEPS * WARMUP_FRACTION)))
print("Optimizer updates per epoch:", STEPS_PER_EPOCH)
print("Fixed total optimizer updates:", MAX_STEPS)
print("Effective WARMUP_STEPS:", WARMUP_STEPS)

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=RMS_NORM_EPS):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = float(eps)

    def forward(self, x):
        input_dtype = x.dtype
        x_float = x.float()
        rms_inv = torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        normalized = (x_float * rms_inv).to(dtype=input_dtype)
        return normalized * self.weight.to(dtype=input_dtype)

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10_000.0):
        super().__init__()
        if head_dim % 2:
            raise ValueError(f"RoPE requires an even head dimension; got {head_dim}.")
        self.head_dim = int(head_dim)
        self.max_seq_len = int(max_seq_len)
        inv_freq = 1.0 / (
            float(base) ** (
                torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim
            )
        )
        positions = torch.arange(max_seq_len, dtype=torch.float32)
        frequencies = torch.outer(positions, inv_freq)
        self.register_buffer(
            "cos_cached",
            frequencies.cos().repeat_interleave(2, dim=-1)[None, None, :, :],
            persistent=False,
        )
        self.register_buffer(
            "sin_cached",
            frequencies.sin().repeat_interleave(2, dim=-1)[None, None, :, :],
            persistent=False,
        )

    @staticmethod
    def rotate_half(x):
        even = x[..., 0::2]
        odd = x[..., 1::2]
        return torch.stack((-odd, even), dim=-1).flatten(-2)

    def forward(self, x, position_offset=0):
        length = x.size(-2)
        start = int(position_offset)
        stop = start + length
        if start < 0 or stop > self.max_seq_len:
            raise ValueError(
                f"RoPE positions [{start}, {stop}) exceed maximum {self.max_seq_len}."
            )
        cos = self.cos_cached[:, :, start:stop].to(device=x.device, dtype=x.dtype)
        sin = self.sin_cached[:, :, start:stop].to(device=x.device, dtype=x.dtype)
        return x * cos + self.rotate_half(x) * sin

class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.1, use_rope=True):
        super().__init__()
        if d_model % n_heads:
            raise ValueError("d_model must be divisible by n_heads.")
        if n_heads % n_kv_heads:
            raise ValueError("n_heads must be divisible by n_kv_heads.")
        self.n_heads = int(n_heads)
        self.n_kv_heads = int(n_kv_heads)
        self.groups_per_kv = self.n_heads // self.n_kv_heads
        self.head_dim = d_model // self.n_heads
        self.dropout = float(dropout)
        self.use_rope = bool(use_rope)
        kv_width = self.n_kv_heads * self.head_dim
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, kv_width)
        self.v_proj = nn.Linear(d_model, kv_width)
        self.out_proj = nn.Linear(d_model, d_model)
        self.rope = (
            RotaryEmbedding(self.head_dim, max_seq_len, ROPE_BASE)
            if self.use_rope else None
        )

    def project_kv(self, key_value_states):
        batch, length, _ = key_value_states.shape
        keys = self.k_proj(key_value_states).view(
            batch, length, self.n_kv_heads, self.head_dim
        ).transpose(1, 2)
        values = self.v_proj(key_value_states).view(
            batch, length, self.n_kv_heads, self.head_dim
        ).transpose(1, 2)
        return keys, values

    def forward(
        self,
        query_states,
        key_value_states=None,
        key_padding_mask=None,
        causal=False,
        past_key_value=None,
        static_key_value=None,
        use_cache=False,
        position_offset=0,
    ):
        batch, query_length, width = query_states.shape
        queries = self.q_proj(query_states).view(
            batch, query_length, self.n_heads, self.head_dim
        ).transpose(1, 2)

        if static_key_value is not None:
            keys, values = static_key_value
        else:
            key_value_states = query_states if key_value_states is None else key_value_states
            keys, values = self.project_kv(key_value_states)
            if self.use_rope:
                keys = self.rope(keys, position_offset=position_offset)
        if self.use_rope:
            queries = self.rope(queries, position_offset=position_offset)

        cached_length = 0 if past_key_value is None else past_key_value[0].size(-2)
        if past_key_value is not None:
            old_keys, old_values = past_key_value
            keys = torch.cat((old_keys, keys), dim=-2)
            values = torch.cat((old_values, values), dim=-2)
        key_length = keys.size(-2)
        expanded_keys = keys.repeat_interleave(self.groups_per_kv, dim=1)
        expanded_values = values.repeat_interleave(self.groups_per_kv, dim=1)

        allowed = None
        if key_padding_mask is not None:
            allowed = key_padding_mask[:, None, None, :].to(dtype=torch.bool)
        if causal:
            query_positions = cached_length + torch.arange(query_length, device=query_states.device)
            key_positions = torch.arange(key_length, device=query_states.device)
            causal_allowed = key_positions[None, :] <= query_positions[:, None]
            causal_allowed = causal_allowed[None, None, :, :]
            allowed = causal_allowed if allowed is None else (allowed & causal_allowed)

        output = F.scaled_dot_product_attention(
            queries,
            expanded_keys,
            expanded_values,
            attn_mask=allowed,
            is_causal=False,
            dropout_p=self.dropout if self.training else 0.0,
        )
        output = output.transpose(1, 2).contiguous().view(batch, query_length, width)
        output = self.out_proj(output)
        if use_cache:
            return output, (keys, values)
        return output

class FeedForward(nn.Module):
    def __init__(self, d_model, hidden, dropout):
        super().__init__()
        self.swiglu = bool(USE_SWIGLU)
        if self.swiglu:
            self.gate_proj = nn.Linear(d_model, hidden, bias=False)
            self.up_proj = nn.Linear(d_model, hidden, bias=False)
            self.down_proj = nn.Linear(hidden, d_model, bias=False)
            self.drop = nn.Dropout(dropout)
        else:
            self.net = nn.Sequential(
                nn.Linear(d_model, hidden),
                nn.GELU(),
                nn.Linear(hidden, d_model),
                nn.Dropout(dropout),
            )

    def forward(self, x):
        if self.swiglu:
            return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))
        return self.net(x)

class SourceCopyHead(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = int(d_model)
        self.query = nn.Linear(d_model, d_model, bias=False)
        self.key = nn.Linear(d_model, d_model, bias=False)
        self.gate = nn.Linear(d_model, 1)
        special = torch.zeros(VOCAB_SIZE, dtype=torch.bool)
        for token_id in SPECIAL_IDS:
            special[int(token_id)] = True
        self.register_buffer("special_mask", special, persistent=False)

    def project_keys(self, memory):
        return self.key(memory)

    def forward(self, hidden, generation_logits, copy_keys, source_ids, source_mask):

        with torch.autocast(device_type="cuda", enabled=False):
            hidden32 = hidden.float()
            batch, steps, _ = hidden32.shape
            scores = torch.matmul(
                self.query(hidden32), copy_keys.float().transpose(-1, -2)
            ) * (self.d_model ** -0.5)
            allowed = source_mask.bool() & ~self.special_mask[source_ids]
            has_copy_target = allowed.any(dim=-1)
            scores = scores.masked_fill(~allowed[:, None, :], -1e9)
            attention = torch.softmax(scores, dim=-1) * has_copy_target[:, None, None].float()
            copy_distribution = torch.zeros(
                batch, steps, self.special_mask.numel(), device=hidden32.device, dtype=torch.float32
            ).scatter_add_(-1, source_ids[:, None, :].expand(-1, steps, -1), attention)
            p_gen = torch.sigmoid(self.gate(hidden32))
            p_gen = torch.where(has_copy_target[:, None, None], p_gen, torch.ones_like(p_gen))
            mixture = p_gen * torch.softmax(generation_logits.float(), dim=-1) + (1.0 - p_gen) * copy_distribution
            return torch.log(mixture.clamp_min(1e-9))

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, ff_hidden, context, dropout):
        super().__init__()
        self.norm_attn = RMSNorm(d_model)
        self.norm_ff = RMSNorm(d_model)
        self.self_attn = GroupedQueryAttention(
            d_model, n_heads, n_kv_heads, context, dropout, use_rope=True
        )
        self.ff = FeedForward(d_model, ff_hidden, dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask):
        attention = self.self_attn(
            self.norm_attn(x), key_padding_mask=key_padding_mask, causal=False
        )
        x = x + self.resid_dropout(attention)
        x = x + self.resid_dropout(self.ff(self.norm_ff(x)))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, ff_hidden, context, dropout):
        super().__init__()
        self.norm_self = RMSNorm(d_model)
        self.norm_cross = RMSNorm(d_model)
        self.norm_ff = RMSNorm(d_model)
        self.self_attn = GroupedQueryAttention(
            d_model, n_heads, n_kv_heads, context, dropout, use_rope=True
        )
        self.cross_attn = GroupedQueryAttention(
            d_model, n_heads, n_kv_heads, context, dropout, use_rope=False
        )
        self.ff = FeedForward(d_model, ff_hidden, dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(
        self,
        x,
        memory,
        memory_padding_mask,
        decoder_padding_mask=None,
        past_key_value=None,
        cross_key_value=None,
        use_cache=False,
        position_offset=0,
    ):
        self_input = self.norm_self(x)
        if use_cache:
            self_output, present = self.self_attn(
                self_input,
                key_padding_mask=decoder_padding_mask,
                causal=True,
                past_key_value=past_key_value,
                use_cache=True,
                position_offset=position_offset,
            )
        else:
            self_output = self.self_attn(
                self_input,
                key_padding_mask=decoder_padding_mask,
                causal=True,
            )
            present = None
        x = x + self.resid_dropout(self_output)
        cross_output = self.cross_attn(
            self.norm_cross(x),
            key_value_states=memory if cross_key_value is None else None,
            key_padding_mask=memory_padding_mask,
            static_key_value=cross_key_value,
            causal=False,
        )
        x = x + self.resid_dropout(cross_output)
        x = x + self.resid_dropout(self.ff(self.norm_ff(x)))
        if use_cache:
            return x, present
        return x

class EncoderDecoderTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model,
        n_heads,
        n_kv_heads,
        encoder_layers,
        decoder_layers,
        ff_hidden,
        context,
        dropout,
    ):
        super().__init__()
        self.vocab_size = int(vocab_size)
        self.d_model = int(d_model)
        self.encoder_max_length = int(context)
        self.decoder_max_length = int(context)
        self.shared_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, n_kv_heads, ff_hidden, context, dropout)
            for _ in range(encoder_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, n_kv_heads, ff_hidden, context, dropout)
            for _ in range(decoder_layers)
        ])
        self.encoder_norm = RMSNorm(d_model)
        self.decoder_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.shared_embedding.weight
        self.copy_head = SourceCopyHead(d_model) if USE_COPY_HEAD else None
        self.apply(self._init_weights)
        nn.init.normal_(self.shared_embedding.weight, mean=0.0, std=0.02)
        if self.copy_head is not None:
            nn.init.constant_(self.copy_head.gate.bias, 2.0)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, RMSNorm):
            nn.init.ones_(module.weight)

    def encode(self, source_ids, source_padding_mask=None):
        if source_ids.size(1) > self.encoder_max_length:
            raise ValueError(
                f"Encoder input {source_ids.size(1)} exceeds {self.encoder_max_length} positions."
            )
        if source_padding_mask is None:
            source_padding_mask = source_ids.ne(PAD_ID)
        x = self.shared_embedding(source_ids)
        for layer in self.encoder_layers:
            if self.training and USE_GRAD_CHECKPOINTING:
                x = checkpoint(
                    lambda hidden, layer=layer: layer(hidden, source_padding_mask),
                    x,
                    use_reentrant=False,
                )
            else:
                x = layer(x, source_padding_mask)
        return self.encoder_norm(x)

    def prepare_cross_key_values(self, memory):
        return tuple(layer.cross_attn.project_kv(memory) for layer in self.decoder_layers)

    def decode(
        self,
        decoder_input_ids,
        memory,
        memory_padding_mask,
        decoder_padding_mask=None,
        past_key_values=None,
        cross_key_values=None,
        use_cache=False,
        copy_keys=None,
        source_ids=None,
    ):
        if decoder_padding_mask is None:
            decoder_padding_mask = decoder_input_ids.ne(PAD_ID)
        if past_key_values is None:
            past_key_values = (None,) * len(self.decoder_layers)
            position_offset = 0
        else:
            if len(past_key_values) != len(self.decoder_layers):
                raise ValueError("Decoder KV cache layer count is incompatible.")
            lengths = {cache[0].size(-2) for cache in past_key_values}
            if len(lengths) != 1:
                raise ValueError("Decoder KV cache lengths do not match.")
            position_offset = lengths.pop()
            use_cache = True
            if decoder_padding_mask.size(1) == decoder_input_ids.size(1) and position_offset:
                cached_keep = torch.ones(
                    (decoder_input_ids.size(0), position_offset),
                    dtype=torch.bool,
                    device=decoder_input_ids.device,
                )
                decoder_padding_mask = torch.cat(
                    (cached_keep, decoder_padding_mask), dim=1
                )
            if decoder_padding_mask.size(1) != position_offset + decoder_input_ids.size(1):
                raise ValueError("Decoder key mask length does not match cached keys.")
        total_length = position_offset + decoder_input_ids.size(1)
        if total_length > self.decoder_max_length:
            raise ValueError(
                f"Decoder length {total_length} exceeds {self.decoder_max_length} positions."
            )

        x = self.shared_embedding(decoder_input_ids)
        next_key_values = [] if use_cache else None
        for index, layer in enumerate(self.decoder_layers):
            layer_past = past_key_values[index]
            layer_cross = cross_key_values[index] if cross_key_values is not None else None
            if self.training and USE_GRAD_CHECKPOINTING and not use_cache:
                x = checkpoint(
                    lambda hidden, enc, layer=layer: layer(
                        hidden, enc, memory_padding_mask, decoder_padding_mask
                    ),
                    x,
                    memory,
                    use_reentrant=False,
                )
            elif use_cache:
                x, present = layer(
                    x,
                    memory,
                    memory_padding_mask,
                    decoder_padding_mask=decoder_padding_mask,
                    past_key_value=layer_past,
                    cross_key_value=layer_cross,
                    use_cache=True,
                    position_offset=position_offset,
                )
                next_key_values.append(present)
            else:
                x = layer(
                    x,
                    memory,
                    memory_padding_mask,
                    decoder_padding_mask=decoder_padding_mask,
                )
        decoder_hidden = self.decoder_norm(x)
        logits = self.lm_head(decoder_hidden)
        if self.copy_head is not None:
            if copy_keys is None or source_ids is None:
                raise ValueError("The copy head needs copy_keys and source_ids.")
            logits = self.copy_head(decoder_hidden, logits, copy_keys, source_ids, memory_padding_mask)
        if use_cache:
            return logits, tuple(next_key_values)
        return logits

    def forward(self, source_ids, decoder_input_ids):
        source_padding_mask = source_ids.ne(PAD_ID)
        decoder_padding_mask = decoder_input_ids.ne(PAD_ID)
        memory = self.encode(source_ids, source_padding_mask)
        copy_keys = self.copy_head.project_keys(memory) if self.copy_head is not None else None
        return self.decode(
            decoder_input_ids,
            memory,
            source_padding_mask,
            decoder_padding_mask=decoder_padding_mask,
            copy_keys=copy_keys,
            source_ids=source_ids,
        )

model_config = {
    "architecture": ARCHITECTURE_NAME,
    "model_type": "encoder_decoder",
    "positional_encoding": "RoPE on encoder/decoder self-attention",
    "normalization": "RMSNorm",
    "rms_norm_eps": RMS_NORM_EPS,
    "rope_base": ROPE_BASE,
    "vocab_size": VOCAB_SIZE,
    "d_model": MODEL_D,
    "attention_type": "grouped_query_attention",
    "cross_attention_type": "grouped_query_attention",
    "query_heads": MODEL_HEADS,
    "kv_heads": MODEL_KV_HEADS,
    "query_heads_per_kv_head": MODEL_HEADS // MODEL_KV_HEADS,
    "head_dim": MODEL_D // MODEL_HEADS,
    "encoder_layers": ENCODER_LAYERS,
    "decoder_layers": DECODER_LAYERS,
    "ff_hidden": MODEL_FF,
    "encoder_max_length": ENCODER_MAX_LENGTH,
    "decoder_max_length": DECODER_MAX_LENGTH,
    "dropout": MODEL_DROPOUT,
    "weight_tied_source_target_output_embeddings": True,
    "feed_forward": "swiglu" if USE_SWIGLU else "gelu_mlp",
    "source_copy_head": bool(USE_COPY_HEAD),
}

def build_model():
    return EncoderDecoderTransformer(
        vocab_size=VOCAB_SIZE,
        d_model=MODEL_D,
        n_heads=MODEL_HEADS,
        n_kv_heads=MODEL_KV_HEADS,
        encoder_layers=ENCODER_LAYERS,
        decoder_layers=DECODER_LAYERS,
        ff_hidden=MODEL_FF,
        context=FIXED_CONTEXT_LENGTH,
        dropout=MODEL_DROPOUT,
    )

def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())

def architecture_audit(model):
    total = count_parameters(model)
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    tied = model.lm_head.weight.data_ptr() == model.shared_embedding.weight.data_ptr()
    return {
        "total_parameters": int(total),
        "trainable_parameters": int(trainable),
        "parameter_millions": total / 1e6,
        "weight_tied_source_target_output_embeddings": bool(tied),
        "learned_absolute_position_parameters": 0,
        "positional_encoding": "RoPE",
        "normalization": "RMSNorm",
        "rms_norm_eps": RMS_NORM_EPS,
        "rope_base": ROPE_BASE,
        "config": model_config,
    }

model_for_audit = build_model()
audit = architecture_audit(model_for_audit)
print(json.dumps(audit, indent=2))
if not audit["weight_tied_source_target_output_embeddings"]:
    raise RuntimeError("Expected tied source, target, and output embeddings.")
if MODEL_D // MODEL_HEADS % 2:
    raise RuntimeError("RoPE requires an even head dimension.")
if not (100_000_000 <= audit["total_parameters"] <= 250_000_000):
    raise RuntimeError(
        f"Parameter count must be within 100M-250M; "
        f"got {audit['total_parameters']:,}."
    )
print(
    f"Parameter count: {audit['total_parameters']:,} "
    f"({audit['parameter_millions']:.3f}M)"
)
print(
    f"Encoder/decoder layers: {ENCODER_LAYERS}/{DECODER_LAYERS}; "
    f"GQA heads: {MODEL_HEADS} query, {MODEL_KV_HEADS} key/value"
)
del model_for_audit
gc.collect()

In [ ]:
if RUN_SMOKE_TEST:
    rope = RotaryEmbedding(
        head_dim=MODEL_D // MODEL_HEADS,
        max_seq_len=BLOCK_SIZE,
        base=ROPE_BASE,
    ).to(DEVICE)
    q_test = torch.randn(
        2, MODEL_HEADS, 32, MODEL_D // MODEL_HEADS, device=DEVICE
    )
    q_rot = rope(q_test)
    assert q_rot.shape == q_test.shape
    assert torch.isfinite(q_rot).all()
    assert (q_rot - q_test).abs().mean().item() > 0
    assert sum(p.numel() for p in rope.parameters()) == 0
    print("RoPE correctness check: PASSED; Q shape:", tuple(q_rot.shape))
    del rope, q_test, q_rot
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
import time

if RUN_SMOKE_TEST:
    SMOKE_SOURCE_LEN = 64
    SMOKE_TARGET_LEN = 32
    smoke_model = build_model().to(DEVICE)
    smoke_model.train()
    sx = torch.randint(4, VOCAB_SIZE, (MICRO_BATCH_SIZE, SMOKE_SOURCE_LEN), device=DEVICE)
    sd = torch.randint(4, VOCAB_SIZE, (MICRO_BATCH_SIZE, SMOKE_TARGET_LEN), device=DEVICE)
    sy = torch.randint(4, VOCAB_SIZE, (MICRO_BATCH_SIZE, SMOKE_TARGET_LEN), device=DEVICE)
    sx[:, 0] = BOS_ID
    sd[:, 0] = BOS_ID
    use_fp16 = DEVICE.type == "cuda" and not torch.cuda.is_bf16_supported()
    amp_dtype = torch.float16 if use_fp16 else torch.bfloat16
    scaler = make_grad_scaler(use_fp16) if DEVICE.type == "cuda" else None
    optimizer = torch.optim.AdamW(
        smoke_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    optimizer.zero_grad(set_to_none=True)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    with torch.autocast(
        device_type="cuda", dtype=amp_dtype, enabled=(DEVICE.type == "cuda")
    ):
        slogits = smoke_model(sx, sd)
        sloss = F.cross_entropy(
            slogits.reshape(-1, VOCAB_SIZE), sy.reshape(-1), ignore_index=PAD_ID
        )
    if scaler is not None and scaler.is_enabled():
        scaler.scale(sloss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(smoke_model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
    else:
        sloss.backward()
        torch.nn.utils.clip_grad_norm_(smoke_model.parameters(), GRAD_CLIP)
        optimizer.step()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - start
    print(f"Smoke source/target lengths: {SMOKE_SOURCE_LEN}/{SMOKE_TARGET_LEN}")
    print(f"Smoke loss: {sloss.item():.4f}; update time: {elapsed:.3f}s")
    if DEVICE.type == "cuda":
        print(f"Peak VRAM: {torch.cuda.max_memory_allocated() / (1024**3):.2f} GB")
        torch.cuda.reset_peak_memory_stats()
    del smoke_model, optimizer, scaler, sx, sd, sy, slogits, sloss
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
def make_loader(dataset, split):
    return DataLoader(
        dataset,
        batch_size=MICRO_BATCH_SIZE,
        shuffle=(split == "train"),
        num_workers=WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=False,
        persistent_workers=(WORKERS > 0),
    )

USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.float16 if USE_AMP else torch.float32
USE_SCALER = USE_AMP

def amp_context():
    return torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP)

def make_scheduler(optimizer, total_steps, warmup_steps):
    def lr_lambda(step):
        if step < warmup_steps:
            return max(1e-8, (step + 1) / max(1, warmup_steps))
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        cosine = 0.5 * (1.0 + math.cos(math.pi * min(1.0, max(0.0, progress))))
        return max(0.05, cosine)
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

@torch.no_grad()
def validation_loss(model, loader, max_batches=MAX_EVAL_BATCHES):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    for batch_idx, (source_ids, decoder_ids, labels) in enumerate(loader):
        if batch_idx >= max_batches:
            break
        source_ids = source_ids.to(DEVICE, non_blocking=True)
        decoder_ids = decoder_ids.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with amp_context():
            logits = model(source_ids, decoder_ids)
            loss_sum = F.cross_entropy(
                logits.reshape(-1, VOCAB_SIZE),
                labels.reshape(-1),
                ignore_index=PAD_ID,
                reduction="sum",
            )
        total_loss += float(loss_sum.item())
        total_tokens += int(labels.ne(PAD_ID).sum().item())
    if total_tokens == 0:
        raise RuntimeError("Validation loader produced no target tokens.")
    return total_loss / total_tokens

def save_checkpoint(
    path, model, optimizer, scheduler, scaler, epoch, step, best_valid_loss, history,
    best_bertscore=None, selection_score=None,
):
    payload = {
        "architecture": ARCHITECTURE_NAME,
        "model_config": model_config,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "epoch": int(epoch),
        "step": int(step),
        "best_valid_loss": float(best_valid_loss),
        "best_checkpoint_bertscore_f1": (
            float(best_bertscore) if best_bertscore is not None else None
        ),
        "selection_bertscore_f1": (
            float(selection_score) if selection_score is not None else None
        ),
        "checkpoint_selection_metric": "greedy_generation_BERTScore_F1",
        "history": history,
        "seed": SEED,
        "vocab_size": VOCAB_SIZE,
        "tokenizer_sha256": TOKENIZER_SHA256,
    }
    torch.save(payload, path)

def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

In [ ]:
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
STEPS_PER_EPOCH = max(1, math.ceil(len(train_pairs) / (MICRO_BATCH_SIZE * GRAD_ACCUM)))
MAX_STEPS = EPOCHS * STEPS_PER_EPOCH
WARMUP_STEPS = max(1, int(round(MAX_STEPS * WARMUP_FRACTION)))
mean_pair_tokens = TRAIN_PAIR_TOKEN_COUNT / max(1, len(train_pairs))
print("GPU:", GPU_NAME)
print("Train source/target pairs:", f"{len(train_pairs):,}")
print("Validation source/target pairs:", f"{len(valid_pairs):,}")
print("Mean source+target tokens per pair:", f"{mean_pair_tokens:.1f}")
print("Tokens per optimizer update (approx.):", f"{mean_pair_tokens * GRAD_ACCUM:,.0f}")
print("Unique train corpus coverage per epoch:", f"{TRAIN_COVERED_TOKEN_COUNT:,}")
print("Fixed epochs:", EPOCHS)
print("Optimizer updates per epoch:", STEPS_PER_EPOCH)
print("Warmup updates:", WARMUP_STEPS)

In [ ]:
def load_selected_model():
    path = BEST_CHECKPOINT_PATH if BEST_CHECKPOINT_PATH.exists() else LATEST_CHECKPOINT_PATH
    if not path.exists():
        raise FileNotFoundError("Train the model first; no model checkpoint was found.")
    payload = torch.load(path, map_location=DEVICE, weights_only=False)
    if payload.get("architecture") != ARCHITECTURE_NAME:
        raise ValueError("Checkpoint architecture does not match this notebook.")
    if payload.get("vocab_size") != VOCAB_SIZE:
        raise ValueError("Checkpoint vocabulary size does not match the tokenizer.")
    if payload.get("tokenizer_sha256") != TOKENIZER_SHA256:
        raise ValueError("Checkpoint tokenizer hash does not match the loaded tokenizer.")
    if payload.get("model_config") != model_config:
        raise ValueError("Checkpoint model configuration does not match this notebook.")
    model = build_model().to(DEVICE)
    model.load_state_dict(payload["model_state_dict"], strict=True)
    model.eval()
    return model, path

def sample_token(logits, temperature=None, top_k=None, top_p=None):
    if temperature is None:
        temperature = TEMPERATURE
    if top_k is None:
        top_k = TOP_K
    if top_p is None:
        top_p = TOP_P
    logits = logits.float().clone()
    for token_id in SPECIAL_IDS:
        if 0 <= int(token_id) < logits.numel():
            logits[int(token_id)] = -float("inf")
    if temperature <= 0:
        return int(torch.argmax(logits))
    logits = logits / temperature
    if top_k is not None and 0 < top_k < logits.numel():
        threshold = torch.topk(logits, top_k).values[-1]
        logits[logits < threshold] = -float("inf")
    if top_p is not None and 0 < top_p < 1.0:
        values, indices = torch.sort(logits, descending=True)
        probabilities = torch.softmax(values, dim=-1)
        cumulative = probabilities.cumsum(-1)
        remove = cumulative > top_p
        remove[1:] = remove[:-1].clone()
        remove[0] = False
        logits[indices[remove]] = -float("inf")
    probabilities = torch.softmax(logits, dim=-1)
    if not torch.isfinite(probabilities).all() or float(probabilities.sum()) <= 0:
        return int(torch.argmax(logits))
    return int(torch.multinomial(probabilities, 1))

@torch.no_grad()
def generate_continuation(
    model,
    prompt,
    target_words=OUTPUT_WORDS,
    max_tokens=None,
    temperature=None,
    top_k=None,
    top_p=None,
):
    model.eval()
    if temperature is None:
        temperature = TEMPERATURE
    if max_tokens is None:
        max_tokens = MAX_GENERATION_TOKENS
    prompt_ids = tokenizer.encode(prompt)
    if not prompt_ids:
        prompt_ids = [UNK_ID]
    source_ids = [BOS_ID] + prompt_ids + [EOS_ID]
    if len(source_ids) > ENCODER_MAX_LENGTH:
        raise RuntimeError(
            f"Prompt needs {len(source_ids)} encoder tokens, limit is {ENCODER_MAX_LENGTH}."
        )
    max_tokens = min(int(max_tokens), DECODER_MAX_LENGTH)
    if max_tokens <= 0:
        raise RuntimeError("Decoder token budget must be positive.")

    source = torch.tensor(source_ids, dtype=torch.long, device=DEVICE)[None, :]
    source_keep = source.ne(PAD_ID)
    with amp_context():
        memory = model.encode(source, source_keep)
        cross_key_values = model.prepare_cross_key_values(memory)
        copy_keys = (
            model.copy_head.project_keys(memory)
            if getattr(model, "copy_head", None) is not None else None
        )

    end_flags = build_word_end_flags(tokenizer)
    generated = []
    completed_words = 0
    pieces_in_current_word = 0
    current_word_has_bytes = False
    past_key_values = None

    for _ in range(max_tokens):
        if USE_KV_CACHE and past_key_values is not None:
            decoder_prefix = [generated[-1]]
        else:
            decoder_prefix = [BOS_ID] + generated
        decoder_input = torch.tensor(
            decoder_prefix, dtype=torch.long, device=DEVICE
        )[None, :]
        with amp_context():
            decode_result = model.decode(
                decoder_input,
                memory,
                source_keep,
                past_key_values=past_key_values if USE_KV_CACHE else None,
                cross_key_values=cross_key_values,
                use_cache=USE_KV_CACHE,
                copy_keys=copy_keys,
                source_ids=source,
            )
        if USE_KV_CACHE:
            logits, past_key_values = decode_result
        else:
            logits = decode_result
        next_logits = logits[0, -1]
        if not current_word_has_bytes:
            next_logits = next_logits.clone()
            next_logits[EOW_ID] = -float("inf")
        next_id = sample_token(
            next_logits, temperature=temperature, top_k=top_k, top_p=top_p
        )
        generated.append(next_id)
        token_payload = tokenizer.payload.get(next_id, b"")

        if end_flags[next_id]:
            if current_word_has_bytes or token_payload:
                completed_words += 1
            current_word_has_bytes = False
            pieces_in_current_word = 0
        else:
            if token_payload:
                current_word_has_bytes = True
            pieces_in_current_word += 1
            if current_word_has_bytes and pieces_in_current_word >= MAX_BPE_TOKENS_PER_WORD:
                generated.append(EOW_ID)
                past_key_values = None
                completed_words += 1
                current_word_has_bytes = False
                pieces_in_current_word = 0

        if completed_words >= target_words:
            text = decode_bpe_ids(tokenizer, generated, skip_special_tokens=True).strip()
            words = text.split()
            if len(words) < target_words:
                raise RuntimeError(
                    f"Word-boundary accounting produced {len(words)} words; expected {target_words}."
                )
            return " ".join(words[:target_words])

    text = decode_bpe_ids(tokenizer, generated, skip_special_tokens=True).strip()
    words = text.split()
    raise RuntimeError(
        f"Generated {len(words)} words (< {target_words}) after {max_tokens} decoder tokens."
    )

def _fit_evaluation_pairs(pairs, limit=None):
    if limit is not None:
        pairs = pairs[:limit]
    if not pairs:
        raise RuntimeError("The requested held-out evaluation partition is empty.")
    fitted_pairs = []
    for prompt, reference in pairs:
        prompt_words = prompt.split()
        while prompt_words:
            candidate = " ".join(prompt_words)
            if len(tokenizer.encode(candidate)) + 2 <= ENCODER_MAX_LENGTH:
                fitted_pairs.append((candidate, reference))
                break
            prompt_words = prompt_words[max(1, len(prompt_words) // 16):]
        if not prompt_words:
            raise RuntimeError("Could not fit a held-out prompt in the encoder context.")
    return fitted_pairs

def checkpoint_selection_pairs():
    return _fit_evaluation_pairs(CHECKPOINT_SELECTION_PAIRS)

def decoding_tuning_pairs(limit=DECODING_TUNING_EXAMPLES):
    return _fit_evaluation_pairs(DECODING_TUNING_PAIRS, limit)

def validation_pairs(limit=LOCAL_EVAL_EXAMPLES):
    return _fit_evaluation_pairs(HELDOUT_VALIDATION_PAIRS, limit)

def lcs_length(a, b):
    if not a or not b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    masks = {}
    for j, token in enumerate(a):
        masks[token] = masks.get(token, 0) | (1 << j)
    state = 0
    for token in b:
        m = masks.get(token, 0)
        x = m | state
        y = (state << 1) | 1
        state = x & ~(x - y)
    return state.bit_count() if hasattr(int, "bit_count") else bin(state).count("1")

def rouge_l_f1(pred_text, ref_text):
    pred = pred_text.split()
    ref = ref_text.split()
    if not pred or not ref:
        return 0.0
    l = lcs_length(pred, ref)
    if not l:
        return 0.0
    precision = l / len(pred)
    recall = l / len(ref)
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)

def ngrams(tokens, n):
    if len(tokens) < n:
        return Counter()
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))

def bleu4(preds, refs):
    order = 4
    clipped = [0] * order
    totals = [0] * order
    pred_len = 0
    ref_len = 0

    for pred, ref in zip(preds, refs):
        pred_len += len(pred)
        ref_len += len(ref)
        for n in range(1, order + 1):
            pn = ngrams(pred, n)
            rn = ngrams(ref, n)
            totals[n - 1] += sum(pn.values())
            for gram, count in pn.items():
                clipped[n - 1] += min(count, rn.get(gram, 0))

    precisions = [
        clipped[i] / totals[i] if totals[i] else 0.0
        for i in range(order)
    ]
    if any(x <= 0 for x in precisions) or pred_len == 0:
        return 0.0

    bp = 1.0 if pred_len > ref_len else math.exp(1.0 - ref_len / pred_len)
    return bp * math.exp(sum(math.log(x) for x in precisions) / order)

def local_bertscore(pred_texts, ref_texts):
    import bert_score
    device_name = "cuda" if torch.cuda.is_available() else "cpu"
    _, _, f1 = bert_score.score(
        pred_texts,
        ref_texts,
        lang="en",
        batch_size=8,
        verbose=False,
        device=device_name,
    )
    return float(f1.mean().item())

def make_checkpoint_bertscore_scorer():
    from bert_score import BERTScorer
    return BERTScorer(
        lang="en",
        batch_size=4,
        device="cpu",
        rescale_with_baseline=False,
    )

def score_with_bertscore_scorer(scorer, predictions, references):
    if len(predictions) != len(references) or not predictions:
        raise ValueError("BERTScore needs equally sized, non-empty prediction/reference lists.")
    _, _, f1 = scorer.score(predictions, references)
    return float(f1.mean().item())

In [ ]:
if BLOCK_SIZE != FIXED_CONTEXT_LENGTH:
    raise RuntimeError(
        f"Expected fixed context {FIXED_CONTEXT_LENGTH}, got BLOCK_SIZE={BLOCK_SIZE}."
    )
max_prompt_tokens = max(int(row["tokens"]) for row in prompt_audit)
if max_prompt_tokens + 2 > ENCODER_MAX_LENGTH:
    raise RuntimeError(
        f"Longest official prompt plus BOS/EOS is {max_prompt_tokens + 2} tokens, "
        f"above encoder limit {ENCODER_MAX_LENGTH}."
    )
if MAX_GENERATION_TOKENS > DECODER_MAX_LENGTH:
    raise RuntimeError(
        f"Generation budget {MAX_GENERATION_TOKENS} exceeds decoder limit "
        f"{DECODER_MAX_LENGTH}."
    )
print("Fixed encoder context:", ENCODER_MAX_LENGTH)
print("Fixed decoder context:", DECODER_MAX_LENGTH)
print("Maximum official prompt tokens:", max_prompt_tokens)
print("Generation token budgets:", GENERATION_TOKEN_BUDGETS)

def generate_128_word_continuation(
    model, prompt, temperature=None, top_k=None, top_p=None
):
    last_error = None
    for token_budget in GENERATION_TOKEN_BUDGETS:
        try:
            text = generate_continuation(
                model, prompt, target_words=OUTPUT_WORDS,
                max_tokens=token_budget, temperature=temperature,
                top_k=top_k, top_p=top_p,
            )
            if len(text.split()) != OUTPUT_WORDS:
                raise RuntimeError(
                    f"Generator returned {len(text.split())} words; expected {OUTPUT_WORDS}."
                )
            return text
        except RuntimeError as exc:
            last_error = exc
            message = str(exc).lower()
            if "< " not in message and "expected" not in message:
                raise
    raise RuntimeError(
        f"Could not produce exactly {OUTPUT_WORDS} words with decoder budgets "
        f"{GENERATION_TOKEN_BUDGETS}. Last error: {last_error}"
    ) from last_error

In [ ]:
def try_load_warm_start(model):
    if not CONTINUE_FROM_EXISTING_CHECKPOINT or not BEST_CHECKPOINT_PATH.exists():
        return False
    try:
        payload = torch.load(BEST_CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
        compatible = (
            payload.get("architecture") == ARCHITECTURE_NAME
            and payload.get("vocab_size") == VOCAB_SIZE
            and payload.get("tokenizer_sha256") == TOKENIZER_SHA256
            and payload.get("model_config") == model_config
        )
        if not compatible:
            print("Existing checkpoint is not this encoder-decoder model; starting from scratch.")
            return False
        model.load_state_dict(payload["model_state_dict"], strict=True)
        print("Warm-started from compatible encoder-decoder checkpoint:", BEST_CHECKPOINT_PATH)
        return True
    except Exception as exc:
        print("Could not load compatible encoder-decoder checkpoint; starting from scratch:", exc)
        return False

def _checkpoint_selection_score(model, scorer):
    pairs = checkpoint_selection_pairs()
    references = [reference for _, reference in pairs]
    predictions = []
    for prompt, _ in tqdm(pairs, desc="Greedy checkpoint-selection generation"):
        predictions.append(
            generate_128_word_continuation(
                model, prompt, temperature=0.0, top_k=0, top_p=1.0
            )
        )
    return score_with_bertscore_scorer(scorer, predictions, references)

def train_model():
    if DEVICE.type != "cuda":
        raise RuntimeError("Select the Kaggle GPU accelerator before training.")
    model = build_model().to(DEVICE)
    warm_started = try_load_warm_start(model)
    run_learning_rate = WARM_START_LEARNING_RATE if warm_started else LEARNING_RATE
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=run_learning_rate, betas=(0.9, 0.95),
        eps=1e-8, weight_decay=WEIGHT_DECAY,
    )
    scheduler = make_scheduler(optimizer, MAX_STEPS, WARMUP_STEPS)
    scaler = make_grad_scaler(USE_SCALER) if USE_SCALER else None
    train_loader = make_loader(train_pairs, "train")
    valid_loader = make_loader(valid_pairs, "valid")
    history = []
    best_valid_loss = validation_loss(model, valid_loader) if warm_started else float("inf")
    best_bertscore = -1.0
    if warm_started:
        print(f"Warm-start validation loss={best_valid_loss:.4f}")

    checkpoint_scorer = make_checkpoint_bertscore_scorer()
    if warm_started:
        model.eval()
        best_bertscore = _checkpoint_selection_score(model, checkpoint_scorer)
        print(f"Warm-start checkpoint-selection BERTScore F1={best_bertscore:.4f}")
        save_checkpoint(
            BEST_CHECKPOINT_PATH, model, optimizer, scheduler, scaler,
            0, 0, best_valid_loss, history,
            best_bertscore=best_bertscore, selection_score=best_bertscore,
        )

    global_step = 0
    tokens_processed = 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        batch_iterator = iter(train_loader)
        batches_in_epoch = len(train_loader)
        epoch_loss_sum = 0.0
        epoch_tokens = 0

        for group_start in range(0, batches_in_epoch, GRAD_ACCUM):
            model.train()
            group_size = min(GRAD_ACCUM, batches_in_epoch - group_start)
            grouped_batches = [next(batch_iterator) for _ in range(group_size)]
            group_tokens = sum(
                int(labels.ne(PAD_ID).sum().item())
                for _, _, labels in grouped_batches
            )
            optimizer.zero_grad(set_to_none=True)
            group_loss_sum = 0.0

            for source_ids, decoder_ids, labels in grouped_batches:
                source_ids = source_ids.to(DEVICE, non_blocking=True)
                decoder_ids = decoder_ids.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)
                with amp_context():
                    logits = model(source_ids, decoder_ids)
                    loss_sum = F.cross_entropy(
                        logits.reshape(-1, VOCAB_SIZE), labels.reshape(-1),
                        ignore_index=PAD_ID, reduction="sum",
                    )
                    loss_for_backward = loss_sum / max(1, group_tokens)
                if scaler is not None and scaler.is_enabled():
                    scaler.scale(loss_for_backward).backward()
                else:
                    loss_for_backward.backward()
                group_loss_sum += float(loss_sum.detach().item())

            if scaler is not None and scaler.is_enabled():
                scaler.unscale_(optimizer)
            grad_norm = float(
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP).item()
            )
            if scaler is not None and scaler.is_enabled():
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            scheduler.step()
            global_step += 1
            tokens_processed += group_tokens
            epoch_loss_sum += group_loss_sum
            epoch_tokens += group_tokens
            mean_group_loss = group_loss_sum / max(1, group_tokens)

            if global_step == 1 or global_step % 100 == 0:
                print(
                    f"epoch={epoch}/{EPOCHS} step={global_step}/{MAX_STEPS} "
                    f"train_loss={mean_group_loss:.4f} "
                    f"lr={optimizer.param_groups[0]['lr']:.3e} "
                    f"grad_norm={grad_norm:.3f} target_tokens={tokens_processed:,}"
                )

            if global_step % EVAL_EVERY_STEPS == 0 or global_step == MAX_STEPS:
                val_loss = validation_loss(model, valid_loader)
                val_ppl = math.exp(min(val_loss, 20.0))
                best_valid_loss = min(best_valid_loss, val_loss)
                history.append({
                    "epoch": epoch, "step": global_step,
                    "train_loss": mean_group_loss,
                    "valid_loss": val_loss, "valid_perplexity": val_ppl,
                    "learning_rate": float(optimizer.param_groups[0]["lr"]),
                    "grad_norm": grad_norm,
                    "target_tokens_processed": tokens_processed,
                })
                save_checkpoint(
                    LATEST_CHECKPOINT_PATH, model, optimizer, scheduler, scaler,
                    epoch, global_step, best_valid_loss, history,
                    best_bertscore=best_bertscore,
                )
                save_json(OUTPUT_DIR / "training_history.json", history)
                score_status = (
                    "pending until epoch-end scoring"
                    if best_bertscore < 0
                    else f"{best_bertscore:.4f}"
                )
                print(
                    f"  validation loss={val_loss:.4f}, perplexity={val_ppl:.2f}, "
                    f"best_loss={best_valid_loss:.4f}, "
                    f"best_selection_BERTScore={score_status}"
                )

        epoch_train_loss = epoch_loss_sum / max(1, epoch_tokens)
        model.eval()
        epoch_bertscore = _checkpoint_selection_score(model, checkpoint_scorer)
        improved = epoch_bertscore > best_bertscore
        if improved:
            best_bertscore = epoch_bertscore
            save_checkpoint(
                BEST_CHECKPOINT_PATH, model, optimizer, scheduler, scaler,
                epoch, global_step, best_valid_loss, history,
                best_bertscore=best_bertscore, selection_score=epoch_bertscore,
            )
        history.append({
            "epoch": epoch, "epoch_complete": True,
            "epoch_train_loss": float(epoch_train_loss),
            "checkpoint_selection_bertscore_f1": float(epoch_bertscore),
            "best_checkpoint_bertscore_f1": float(best_bertscore),
            "best_checkpoint_updated": bool(improved),
        })
        save_checkpoint(
            LATEST_CHECKPOINT_PATH, model, optimizer, scheduler, scaler,
            epoch, global_step, best_valid_loss, history,
            best_bertscore=best_bertscore, selection_score=epoch_bertscore,
        )
        save_json(OUTPUT_DIR / "training_history.json", history)
        print(
            f"Epoch {epoch}/{EPOCHS} complete | train_loss={epoch_train_loss:.4f} | "
            f"selection BERTScore F1={epoch_bertscore:.4f} | "
            f"best={best_bertscore:.4f} ({'updated' if improved else 'unchanged'})"
        )

    if not BEST_CHECKPOINT_PATH.exists():
        raise RuntimeError("No best checkpoint was written; inspect training and held-out data.")
    summary = {
        "architecture": ARCHITECTURE_NAME,
        "model_config": model_config,
        "parameters": int(count_parameters(model)),
        "epochs_completed": int(EPOCHS),
        "warm_started_from_compatible_encoder_decoder_checkpoint": bool(warm_started),
        "learning_rate": float(run_learning_rate),
        "steps_completed": int(global_step),
        "target_tokens_processed_this_run": int(tokens_processed),
        "unique_train_corpus_tokens_covered_per_epoch": int(TRAIN_COVERED_TOKEN_COUNT),
        "source_target_token_presentations_per_epoch": int(TRAIN_PAIR_TOKEN_COUNT),
        "train_pairs": int(len(train_pairs)),
        "best_valid_loss": float(best_valid_loss),
        "best_validation_perplexity": float(math.exp(min(best_valid_loss, 20.0))),
        "checkpoint_selection_metric": "greedy_generation_BERTScore_F1",
        "best_checkpoint_bertscore_f1": float(best_bertscore),
        "checkpoint_selection_examples": int(len(CHECKPOINT_SELECTION_PAIRS)),
        "best_checkpoint": str(BEST_CHECKPOINT_PATH),
        "latest_checkpoint": str(LATEST_CHECKPOINT_PATH),
        "training_complete": True,
    }
    save_json(OUTPUT_DIR / "training_summary.json", summary)
    del model, optimizer, scheduler, scaler, train_loader, valid_loader
    del batch_iterator, checkpoint_scorer
    gc.collect()
    torch.cuda.empty_cache()
    return summary

if RUN_TRAINING:
    training_summary = train_model()
    print(json.dumps(training_summary, indent=2))
else:
    summary_path = OUTPUT_DIR / "training_summary.json"
    training_summary = (
        json.loads(summary_path.read_text(encoding="utf-8"))
        if summary_path.exists() else None
    )
    if training_summary is None:
        print("No completed training summary found. Set RUN_TRAINING=True to train the model.")

In [ ]:
TRAINING_COMPLETE_FOR_OUTPUTS = bool(
    training_summary is not None and training_summary.get("training_complete", False)
)

if RUN_DECODING_SWEEP and TRAINING_COMPLETE_FOR_OUTPUTS:
    sweep_pairs = decoding_tuning_pairs()
    sweep_model, sweep_checkpoint = load_selected_model()
    sweep_predictions = {}

    for temperature in DECODING_TEMPERATURES:
        key = str(temperature)
        outputs = []
        for prompt, _ in tqdm(
            sweep_pairs,
            desc=f"Validation decoding sweep (temperature={temperature})",
        ):
            outputs.append(
                generate_128_word_continuation(
                    sweep_model, prompt, temperature=temperature
                )
            )
        sweep_predictions[key] = outputs

    del sweep_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    sweep_results = {}
    sweep_references = [reference for _, reference in sweep_pairs]
    for key, outputs in sweep_predictions.items():
        sweep_results[key] = {
            "BERTScore_F1": local_bertscore(outputs, sweep_references),
            "ROUGE_L_F1": float(np.mean([
                rouge_l_f1(pred, ref)
                for pred, ref in zip(outputs, sweep_references)
            ])),
            "BLEU_4": bleu4(
                [pred.split() for pred in outputs],
                [ref.split() for ref in sweep_references],
            ),
        }

    best_temperature_key = max(
        sweep_results,
        key=lambda key: sweep_results[key]["BERTScore_F1"],
    )
    TEMPERATURE = float(best_temperature_key)
    sweep_payload = {
        "checkpoint": str(sweep_checkpoint),
        "examples": len(sweep_pairs),
        "selected_temperature_by_BERTScore_F1": TEMPERATURE,
        "metrics_by_temperature": sweep_results,
        "predictions_by_temperature": sweep_predictions,
    }
    save_json(OUTPUT_DIR / "local_eval" / "decoding_temperature_sweep.json", sweep_payload)
    print(json.dumps({
        "selected_temperature_by_BERTScore_F1": TEMPERATURE,
        "metrics_by_temperature": sweep_results,
    }, indent=2))
elif RUN_DECODING_SWEEP:
    print("Decoding sweep skipped: train the model first.")

In [ ]:
TRAINING_COMPLETE_FOR_OUTPUTS = bool(
    training_summary is not None and training_summary.get("training_complete", False)
)

if RUN_LOCAL_GENERATION_EVAL and TRAINING_COMPLETE_FOR_OUTPUTS:
    model, checkpoint_path = load_selected_model()
    pairs = validation_pairs()

    local_eval_dir = OUTPUT_DIR / "local_eval"
    local_eval_dir.mkdir(parents=True, exist_ok=True)
    truth = {f"val_{i+1:04d}": {"text": ref} for i, (_, ref) in enumerate(pairs)}
    predictions = {}
    examples = {}

    for i, (prompt, reference) in enumerate(tqdm(pairs, desc="Local held-out generation")):
        prediction = generate_128_word_continuation(model, prompt)
        key = f"val_{i+1:04d}"
        predictions[key] = {"text": prediction}
        examples[key] = {"prompt": prompt, "reference": reference, "prediction": prediction}

    save_json(local_eval_dir / "truth.json", truth)
    save_json(local_eval_dir / "prediction.json", predictions)
    save_json(local_eval_dir / "examples.json", examples)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    pred_texts = [predictions[key]["text"].strip() for key in truth]
    ref_texts = [truth[key]["text"].strip() for key in truth]
    local_metrics = {
        "architecture": ARCHITECTURE_NAME,
        "checkpoint": str(checkpoint_path),
        "examples": len(truth),
        "BERTScore_F1": local_bertscore(pred_texts, ref_texts),
        "ROUGE_L_F1": float(np.mean([rouge_l_f1(p, r) for p, r in zip(pred_texts, ref_texts)])),
        "BLEU_4": bleu4([p.split() for p in pred_texts], [r.split() for r in ref_texts]),
    }
    save_json(local_eval_dir / "metrics.json", local_metrics)
    print(json.dumps(local_metrics, indent=2))
elif RUN_LOCAL_GENERATION_EVAL:
    print("Local generation evaluation skipped: train the model first.")

In [ ]:
def validate_prediction_file(path, expected_ids):
    path = Path(path)

    def pairs_no_duplicates(pairs):
        result = {}
        duplicates = []
        for key, value in pairs:
            if key in result:
                duplicates.append(key)
            result[key] = value
        if duplicates:
            raise ValueError(f"Duplicate JSON keys: {duplicates[:10]}")
        return result

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(
            f,
            object_pairs_hook=pairs_no_duplicates,
            parse_constant=lambda value: (_ for _ in ()).throw(ValueError(f"Non-standard JSON value: {value}")),
        )

    if not isinstance(data, dict):
        raise ValueError("prediction.json must be a JSON object.")
    if set(data) != set(expected_ids):
        raise ValueError(
            f"ID mismatch: missing={len(set(expected_ids)-set(data))}, extra={len(set(data)-set(expected_ids))}"
        )

    for key in expected_ids:
        rec = data[key]
        if not isinstance(rec, dict) or "text" not in rec:
            raise ValueError(f"Invalid record for ID {key!r}")
        text = rec["text"]
        if not isinstance(text, str) or not text.strip():
            raise ValueError(f"Empty/non-string text for ID {key!r}")
        n_words = len(text.split())
        if n_words != OUTPUT_WORDS:
            raise ValueError(f"ID {key} contains {n_words} words; expected {OUTPUT_WORDS}")

    return data

def write_submission(predictions):
    sub_dir = OUTPUT_DIR / "submission_encoder_decoder_gqa_rmsnorm_t4_reused_bpe32k"
    sub_dir.mkdir(parents=True, exist_ok=True)
    json_path = sub_dir / "prediction.json"
    payload = {key: {"text": predictions[key]} for key in official_ids}
    save_json(json_path, payload)
    validate_prediction_file(json_path, official_ids)

    zip_path = OUTPUT_DIR / "submission_phase1_encoder_decoder_gqa_rmsnorm_t4_reused_bpe32k.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(json_path, arcname="prediction.json")

    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()
        if names != ["prediction.json"]:
            raise ValueError(f"Invalid ZIP structure: {names}")

    digest = hashlib.sha256(json_path.read_bytes()).hexdigest()
    return json_path, zip_path, digest

if RUN_OFFICIAL_GENERATION and TRAINING_COMPLETE_FOR_OUTPUTS:

    if "model" in globals():
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    model, selected_checkpoint = load_selected_model()
    partial_path = OUTPUT_DIR / "official_generation_progress_encoder_decoder_gqa_rmsnorm_t4_reused_bpe32k.json"
    valid_ids = set(official_ids)

    checkpoint_file = Path(selected_checkpoint)
    checkpoint_stat = checkpoint_file.stat()
    progress_signature = {
        "checkpoint": str(checkpoint_file.resolve()),
        "checkpoint_size": int(checkpoint_stat.st_size),
        "checkpoint_mtime_ns": int(checkpoint_stat.st_mtime_ns),
        "tokenizer_sha256": TOKENIZER_SHA256,
        "output_words": int(OUTPUT_WORDS),
    }

    def _valid_prediction_map(values):
        result = {}
        if not isinstance(values, dict):
            return result
        for key, value in values.items():
            key = str(key)
            text = value.get("text") if isinstance(value, dict) else value
            if (
                key in valid_ids
                and isinstance(text, str)
                and len(text.split()) == OUTPUT_WORDS
            ):
                result[key] = text
        return result

    in_memory_predictions = globals().get("predictions")
    predictions = {}
    if partial_path.exists():
        try:
            saved_progress = json.loads(partial_path.read_text(encoding="utf-8"))
            if saved_progress.get("signature") == progress_signature:
                predictions.update(_valid_prediction_map(saved_progress.get("predictions", {})))
            else:
                print("Ignoring partial predictions from a different checkpoint/configuration.")
        except Exception as exc:
            print("Could not read the prior partial prediction file; starting from in-memory results:", exc)

    predictions.update(_valid_prediction_map(in_memory_predictions))

    def _save_official_progress():
        payload = {
            "signature": progress_signature,
            "predictions": {
                key: {"text": predictions[key]}
                for key in official_ids
                if key in predictions
            },
        }
        temporary_path = partial_path.with_suffix(".tmp")
        temporary_path.write_text(
            json.dumps(payload, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        temporary_path.replace(partial_path)

    _save_official_progress()
    print(f"Resuming with {len(predictions)}/{len(official_ids)} valid predictions already saved.")

    for item in tqdm(test_examples, desc="Official Phase-1 generation"):
        test_id = str(item["id"])
        if test_id in predictions:
            continue
        try:
            predictions[test_id] = generate_128_word_continuation(
                model,
                item["prompt"],
                temperature=TEMPERATURE,
            )
        except Exception as exc:
            _save_official_progress()
            raise RuntimeError(
                f"Official generation stopped at test ID {test_id}; "
                f"saved {len(predictions)}/{len(official_ids)} predictions to {partial_path}. "
                f"Resume by rerunning this cell after fixing the reported issue."
            ) from exc

        if len(predictions) % 10 == 0:
            _save_official_progress()

    _save_official_progress()
    missing_ids = [key for key in official_ids if key not in predictions]
    if missing_ids:
        raise RuntimeError(f"Generation incomplete; missing {len(missing_ids)} official IDs.")

    prediction_path, zip_path, prediction_sha256 = write_submission(predictions)
    print("prediction.json:", prediction_path)
    print("submission ZIP:", zip_path)
    print("prediction SHA256:", prediction_sha256)

    save_json(OUTPUT_DIR / "submission_manifest.json", {
        "architecture": ARCHITECTURE_NAME,
        "checkpoint": str(selected_checkpoint),
        "num_predictions": len(predictions),
        "first_ids": official_ids[:10],
        "last_ids": official_ids[-10:],
        "prediction_sha256": prediction_sha256,
        "prediction_json": str(prediction_path),
        "submission_zip": str(zip_path),
        "partial_progress": str(partial_path),
        "generation_retry_budgets": list(GENERATION_TOKEN_BUDGETS),
        "inference_context": int(BLOCK_SIZE),
    })

    del model, predictions
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
elif RUN_OFFICIAL_GENERATION:
    print("Official generation skipped: train the model first.")

In [ ]:
manifest = {
    "dataset": DATASET_ID,
    "config": DATASET_CONFIG,
    "seed": SEED,
    "architecture": model_config,
    "architecture_name": ARCHITECTURE_NAME,
    "attention_type": "grouped_query_attention",
    "encoder_layers": ENCODER_LAYERS,
    "decoder_layers": DECODER_LAYERS,
    "query_heads": MODEL_HEADS,
    "kv_heads": MODEL_KV_HEADS,
    "positional_encoding": "RoPE on encoder and decoder self-attention",
    "normalization": "RMSNorm",
    "rms_norm_eps": RMS_NORM_EPS,
    "rope_base": ROPE_BASE,
    "parameters": int(audit["total_parameters"]),
    "parameter_millions": float(audit["parameter_millions"]),
    "tokenizer": str(TOKENIZER_PATH),
    "packaged_tokenizer": str(PACKAGED_TOKENIZER_PATH),
    "tokenizer_sha256": TOKENIZER_SHA256,
    "vocab_size": VOCAB_SIZE,
    "train_documents": int(TRAIN_DOCUMENT_COUNT),
    "valid_documents": int(VALID_DOCUMENT_COUNT),
    "document_counts_by_source": stream_split_document_counts,
    "train_pairs": int(len(train_pairs)),
    "valid_pairs": int(len(valid_pairs)),
    "train_pair_tokens_per_epoch": int(TRAIN_PAIR_TOKEN_COUNT),
    "unique_train_corpus_tokens_covered_per_epoch": int(TRAIN_COVERED_TOKEN_COUNT),
    "requested_unique_train_token_budget": int(TRAIN_TOKEN_BUDGET),
    "required_unique_train_pair_coverage": int(REQUIRED_TRAIN_COVERAGE_TOKENS),
    "valid_pair_tokens": int(VALID_PAIR_TOKEN_COUNT),
    "pair_limits": {
        "source_tokens_max": TRAIN_SOURCE_TOKENS,
        "target_tokens_max": TRAIN_TARGET_TOKENS,
        "source_words_max": TRAIN_SOURCE_WORDS,
        "target_words_exact": TRAIN_TARGET_WORDS,
        "minimum_source_words": MIN_SOURCE_WORDS,
        "stride_words": PAIR_STRIDE_WORDS,
        "over_cap_pairs_skipped_train": train_pairs_skipped_by_source,
        "over_cap_pairs_skipped_validation": valid_pairs_skipped_by_source,
    },
    "heldout_partition_sizes": {
        "checkpoint_selection": len(CHECKPOINT_SELECTION_PAIRS),
        "decoding_tuning": len(DECODING_TUNING_PAIRS),
        "final_local_report": len(HELDOUT_VALIDATION_PAIRS),
    },
    "context": {
        "encoder_max_length": int(ENCODER_MAX_LENGTH),
        "decoder_max_length": int(DECODER_MAX_LENGTH),
        "max_official_prompt_tokens": int(max(prompt_lengths)),
        "estimated_completion_tokens": int(CONTEXT_ESTIMATED_COMPLETION_TOKENS),
        "generation_token_budget": int(MAX_GENERATION_TOKENS),
        "encoder_and_decoder_contexts_are_independent": True,
    },
    "training": {
        "epochs": EPOCHS,
        "steps": MAX_STEPS,
        "micro_batch_size": MICRO_BATCH_SIZE,
        "gradient_accumulation": GRAD_ACCUM,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "gradient_clip": GRAD_CLIP,
        "evaluation_every_steps": EVAL_EVERY_STEPS,
        "max_validation_batches": MAX_EVAL_BATCHES,
        "raw_train_token_budget": TRAIN_TOKEN_BUDGET,
        "required_unique_train_coverage_tokens": REQUIRED_TRAIN_COVERAGE_TOKENS,
        "train_pair_token_budget": TRAIN_PAIR_TOKEN_BUDGET,
        "validation_pair_token_budget": VALID_PAIR_TOKEN_BUDGET,
        "checkpoint_selection_metric": "greedy_generation_BERTScore_F1",
        "best_checkpoint_bertscore_f1": (
            training_summary.get("best_checkpoint_bertscore_f1")
            if training_summary else None
        ),
        "use_gradient_checkpointing": USE_GRAD_CHECKPOINTING,
    },
    "generation": {
        "decoder_kv_cache_enabled": USE_KV_CACHE,
        "output_words": OUTPUT_WORDS,
        "generation_token_budgets": list(GENERATION_TOKEN_BUDGETS),
        "temperature": TEMPERATURE,
        "top_k": TOP_K,
        "top_p": TOP_P,
        "temperature_candidates": DECODING_TEMPERATURES,
    },
    "official_test_count": len(official_ids),
    "official_test_ids": official_ids,
    "local_metrics": local_metrics if "local_metrics" in globals() else None,
    "training_summary": training_summary,
    "output_dir": str(OUTPUT_DIR),
}
save_json(OUTPUT_DIR / "phase1_run_manifest.json", manifest)
print("Saved:", OUTPUT_DIR / "phase1_run_manifest.json")

In [ ]:
import shutil
import os

archive_base = str(Path.cwd() / OUTPUT_DIR.name)
zip_path = shutil.make_archive(
    archive_base,
    "zip",
    root_dir=str(Path.cwd()),
    base_dir=OUTPUT_DIR.name,
)
print("Created:", zip_path)
print("Exists:", os.path.exists(zip_path))
print("Size (MB):", round(os.path.getsize(zip_path) / (1024**2), 2))
print("Packaged tokenizer:", PACKAGED_TOKENIZER_PATH.exists())
print("Best checkpoint:", BEST_CHECKPOINT_PATH.exists())

In [ ]:
from pathlib import Path
import os, torch
from IPython.display import FileLink, display

CKPT = Path("/kaggle/working/phase1_encoder_decoder_gqa_rmsnorm_t4_reused_bpe32k_outputs/best_t4_reused_bpe32k_checkpoint.pt")
assert CKPT.exists(), f"Not found: {CKPT}"
print(f"Full checkpoint: {CKPT.stat().st_size/1e9:.2f} GB")

display(FileLink(str(CKPT.relative_to("/kaggle/working"))))

payload = torch.load(CKPT, map_location="cpu", weights_only=False)
slim = {k: payload[k] for k in
        ("architecture", "model_config", "model_state_dict", "vocab_size", "tokenizer_sha256")
        if k in payload}
SLIM = CKPT.with_name("best_weights_only.pt")
torch.save(slim, SLIM)
print(f"Weights-only copy: {SLIM.stat().st_size/1e9:.2f} GB")
display(FileLink(str(SLIM.relative_to("/kaggle/working"))))